# Road Anomaly Segmentation — Team Template Notebook

**Status: template, not solution.** Every modelling cell raises
`NotImplementedError` until its owner implements it; the notebook stops at the
first stub.

RoadAnomaly is evaluation-only. No weight update, early stopping, threshold,
model choice or registry alias is fitted on its 60 frames. Training uses the
normal-road corpus; the test split opens once, in Section 4.

**One running analogy.** We are teaching a night-shift road inspector to circle
anything that does not belong on the road. The photographs are the data, the
split is an exam in a sealed envelope, and the threshold is how suspicious
something must look before it is circled. The autoencoder is an inspector who
has only seen normal roads.

**Scope Lock.** Only `deeplabv3plus`, `autoencoder` and `unet` exist here, with
the metrics and explanation methods listed in Sections 3 and 4. XAI is not a
fourth network.

## Ownership map

| Area | Owner(s) |
|---|---|
| 1. Dataset comprehension, 2. Data preparation (splits, loaders, augmentation, evaluation protocol) | **Todos** (shared; protocol signed by all three pairs) |
| DeepLabV3+: architecture, input pipeline, ASPP/dilated-convolution justification | **Ines** |
| DeepLabV3+: pretrained encoder selection/configuration, training, fine-tuning, hyperparameters, experiment tracking | **Emna** |
| Autoencoder: architecture, reconstruction function, normal-image preparation, training, reconstruction | **Khaled** |
| Autoencoder: error maps, per-pixel aggregation, threshold chosen on validation, post-processing, false-positive/false-negative study, AUROC and AUPRO | **Adrian** |
| CNN baseline: encoder-decoder, tensor preparation, first training, baseline metrics | **Hejer** |
| U-Net: architecture, skip connections, training, optimization, early stopping, best-model saving, Dice/IoU/HD95 | **Walid** |
| XAI: DeepLabV3+ explanations and comparison | **Ines · Emna** |
| XAI: error maps and interpretation | **Khaled · Adrian** |
| XAI: U-Net explanations and comparative analysis | **Hejer · Walid** |
| Deployment helpers: latency/FPS and checkpoint-reload check | **Khaled · Adrian** |
| Final comparison table, selection rationale, checklist | **Todos** |

Marker convention: a cell starts with `# OWNER: Adrian`; a task is
`# TODO[Adrian]: <one concrete task>`; shared work uses `# OWNER: Todos` and
`# TODO[Todos]: ...`.

Ownership is **coordination responsibility**, not exclusive authorship. Every
member still validates the final report and the demo, and signs the protocol in
Sections 1, 2 and 4.

## How to run

1. From the repository root, install `requirements/dl.txt` in your Jupyter
   environment and start Jupyter.
2. Read Section 0.1, then fill every `None` in `CFG` that your run needs.
3. Run Section 0.1, then Sections 1 and 2 in order.
4. Set `CFG.approach` to the model you own; run that model's cells in Section 3.
5. Run Section 4 only after the threshold is fixed on validation. The test split
   is scored once.
6. Clear outputs before committing; the committed notebook has no outputs.

A cell that stops with `NotImplementedError("TODO[Todos]")` is the intended state.
Keep the contract asserts: they turn a silent shape or split error into a
readable failure. Do not replace one with a print statement.

## 0.1 Setup and reproducibility

**Goal.** Pin the environment, seed, configuration and run folder before any data
or model work.

**Why it matters.** A number without an environment, seed and run folder cannot
be reproduced.

**Feynman check.** The inspector signs in with a badge, writes the date in the
logbook and takes a numbered case folder. Without those three, the shift report
is just a story.

**Watch out.** Absolute paths work on one laptop and fail in CI. A different
TensorFlow or Keras version can move a metric without any code change.

**Your task.**
- [ ] [Todos] Confirm the printed Python and TensorFlow versions match `requirements/dl.txt`.
- [ ] [Todos] Fill every `None` in `CFG` and keep the paths repository-relative.
- [ ] [Todos] Create the run folder and use it for checkpoints and records.

In [ ]:
# OWNER: Todos
# TODO[Todos]: confirm these versions match requirements/dl.txt before a reported run.
from __future__ import annotations

import json
import platform
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path
from typing import Any, Literal

import numpy as np
import tensorflow as tf
from tensorflow import keras

print(f"Python     : {platform.python_version()}")
print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")
print(f"NumPy      : {np.__version__}")

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs       : {[gpu.name for gpu in gpus] or 'none detected (CPU execution)'}")

In [ ]:
# OWNER: Todos
def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository root without hard-coding an absolute path.

    Inputs: optional starting directory; defaults to the current directory.
    Outputs: the nearest directory containing `pyproject.toml` and
    `data/frame_list.json`.
    Contract: the returned directory is absolute and contains both markers.
    Pitfall: a notebook launched from `dl/notebooks/` has a different working
    directory from one launched at the repository root.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "data" / "frame_list.json"
        ).is_file():
            return candidate
    raise FileNotFoundError(
        "Start Jupyter inside the repository, or pass a directory below its root."
    )


REPO_ROOT = find_repo_root()
assert (REPO_ROOT / "data" / "frame_list.json").is_file(), (
    "find_repo_root returned a directory without the dataset manifest"
)


def repo_path(relative: Path | str) -> Path:
    """Resolve a repository-relative path and refuse to escape the repository.

    Inputs: a relative path or string from `CFG`.
    Outputs: an absolute path below `REPO_ROOT`.
    Contract: the resolved path is `REPO_ROOT` or one of its descendants.
    Pitfall: `..` in a config value can silently point outside the checkout.
    """
    candidate = (REPO_ROOT / Path(relative)).resolve()
    if candidate != REPO_ROOT and REPO_ROOT not in candidate.parents:
        raise ValueError(f"{relative!r} escapes the repository at {REPO_ROOT}")
    return candidate


print(f"Repository : {REPO_ROOT}")

In [ ]:
# OWNER: Todos
# TODO[Todos]: every None below is a decision you must make and record.
@dataclass
class Config:
    """Runtime view of one run; the published YAML lives in `configs/`."""

    data_dir: Path = Path("data")  # repository-relative; no absolute paths
    frames_dir: Path = Path("data/frames")
    frame_list: Path = Path("data/frame_list.json")
    normal_road_manifest: Path | None = None
    split_manifest: Path | None = None
    runs_dir: Path = Path("artifacts/runs")
    experiments_dir: Path = Path("docs/experiments")
    checkpoint_path: Path | None = None
    approach: Literal["deeplabv3plus", "autoencoder", "unet"] | None = None
    image_size: tuple[int, int] | None = None
    batch_size: int | None = None
    epochs: int | None = None
    learning_rate: float | None = None
    seed: int = 1337
    threshold_criterion: str | None = None
    boundary_iou_band: float | None = None
    aupro_fpr_cap: float | None = None
    freeze_encoder: bool | None = None
    reload_tolerance: float = 1e-6
    postprocess_min_area: int | None = None
    multiclass_labels_verified: bool = False
    segmentation_num_classes: int | None = None
    empty_mask_policy: str | None = None
    xai_target_layer: str | None = None
    xai_target_pixel: tuple[int, int] | None = None
    xai_baseline: np.ndarray | None = None
    xai_baseline_name: str | None = None
    xai_ig_steps: int | None = None
    xai_faithfulness_steps: int | None = None
    xai_region_size: int | None = None
    background_label: int | None = None  # set after the Section 1 audit
    anomaly_label: int | None = None
    ignore_labels: tuple[int, ...] | None = None  # None = not audited; () = none

In [ ]:
# OWNER: Todos
CFG = Config()
assert CFG.approach in {None, "deeplabv3plus", "autoencoder", "unet"}, (
    "CFG.approach may only be one of the three Scope Lock values"
)
print(f"Active approach: {CFG.approach}")

In [ ]:
# OWNER: Todos
keras.utils.set_random_seed(CFG.seed)
# Deterministic kernels are slower; the alternative is an unrepeatable number.
tf.config.experimental.enable_op_determinism()

assert isinstance(CFG.seed, int) and CFG.seed >= 0, (
    "CFG.seed must be a non-negative integer; a random seed cannot be reported"
)
print(f"Seed: {CFG.seed} (record it in the experiment record)")

In [ ]:
# OWNER: Todos
def create_run_dir(cfg: Config, *, created_at: datetime | None = None) -> Path:
    """Create a unique run folder for checkpoints, logs, figures and records.

    Inputs: the configuration and an optional UTC timestamp.
    Outputs: `artifacts/runs/<UTC>_<approach>_seed<seed>/` with the subfolders
    `checkpoints/`, `logs/`, `figures/` and `records/`.
    Contract: the folder is new and exists after the call.
    Pitfall: two runs in the same second collide; never reuse a run folder, because
    the record would then describe more than one model.
    """
    stamp = (created_at or datetime.now(UTC)).strftime("%Y%m%dT%H%M%SZ")
    approach = cfg.approach or "unconfigured"
    run_dir = repo_path(cfg.runs_dir) / f"{stamp}_{approach}_seed{cfg.seed}"
    for child in ("checkpoints", "logs", "figures", "records"):
        (run_dir / child).mkdir(parents=True, exist_ok=False)
    return run_dir


RUN_DIR = create_run_dir(CFG) if CFG.approach is not None else None
assert RUN_DIR is None or RUN_DIR.is_dir(), (
    "create_run_dir must return an existing directory; check the runs path"
)
print(f"Run folder: {RUN_DIR}")

## 1. Dataset Comprehension

**Goal.** Establish what the 60 evaluation frames contain and freeze the binary
target before any model sees them.

**Why it matters.** If the label audit is skipped, an ignored value is counted as
background and every later metric is wrong.

**Feynman check.** The inspector reads the case file first: how many photographs,
what is circled, which have no circle. Only then can "does not belong" become a
rule.

**Watch out.** A colour mask is a visual aid; the semantic PNG is the label
source. Duplicate scenes and unreadable frames must be found now, not after
scoring.

**Your task.**
- [ ] [Todos] Run the inventory, label audit, coverage and quality checks.
- [ ] [Todos] Resolve every quality flag or record why it is accepted.
- [ ] [Todos] Fill the source/licence fields, write the limitations, and set the three label fields in `CFG`.

### 1.1 Source and licence

Complete these fields before publishing any frame. The attribution in
`data/credits.txt` must travel with every figure.

| Field | Value |
|---|---|
| Dataset release / version | TODO[Todos] |
| Source and retrieval date | TODO[Todos] |
| Image licence per frame | TODO[Todos] — see `data/credits.txt` |
| Annotation licence | TODO[Todos] |
| Redistribution constraints | TODO[Todos] |
| Required citation | TODO[Todos] |

> Write your observation here.

### 1.2 Inventory, label audit and quality

Run the inventory first, then the semantic label audit. The quality checks come
last because they need the frame list the inventory produces.

In [ ]:
# OWNER: Todos
def audit_inventory(frames_dir: Path, frame_list_path: Path) -> dict[str, Any]:
    """Inventory frames and label directories before any split or model work.

    Inputs: absolute frames directory and the JSON frame list.
    Outputs: `frame_names` (ordered list), `frame_count`, `label_dir_count`,
    `missing_labels`, `orphan_label_dirs`, `unreadable`, `sizes`, `modes`.
    Contract: `frame_count == len(frame_names)`; the three problem fields are
    lists of strings; every listed frame has a readable image and one label dir.
    Pitfall: a frame without a label directory, or a label directory without a
    frame, breaks pairing later. Report both; never silently drop either.
    """
    # Hint: json.load the frame list, then compare frames_dir.glob("*.webp") with
    # "*.labels". Open each image with PIL to catch unreadable files.
    raise NotImplementedError("TODO[Todos]: implement the inventory audit")


inventory = audit_inventory(repo_path(CFG.frames_dir), repo_path(CFG.frame_list))
assert {
    "frame_names",
    "frame_count",
    "missing_labels",
    "orphan_label_dirs",
    "unreadable",
} <= set(inventory), "inventory must expose the documented keys"
assert inventory["frame_count"] == len(inventory["frame_names"]), (
    "frame_count must equal the number of listed frames"
)
assert not inventory["missing_labels"], (
    f"frames without label directories: {inventory['missing_labels'][:5]}"
)
assert not inventory["orphan_label_dirs"], (
    f"label directories without frames: {inventory['orphan_label_dirs'][:5]}"
)
assert not inventory["unreadable"], f"unreadable frames: {inventory['unreadable'][:5]}"

In [ ]:
# OWNER: Todos
def audit_labels(frames_dir: Path, frame_names: list[str]) -> dict[str, Any]:
    """Audit the semantic label vocabulary and per-frame pixel counts.

    Inputs: frames directory and the ordered frame names.
    Outputs: `unique_values` (set[int]), `value_counts` (frame -> {value: count}),
    `empty_masks` (frames with only one value), `instance_frames` and
    `semantic_only_frames`.
    Contract: `value_counts` covers every frame exactly once; every count is a
    non-negative integer; `unique_values` is the union of the observed values.
    Pitfall: read `labels_semantic.png`, not the colour PNG. A colour image has
    hundreds of values and would invent labels that were never annotated.
    """
    # Hint: open each labels_semantic.png with PIL, convert to a NumPy array and
    # use np.unique(..., return_counts=True). Keep counts per frame, not only a
    # pooled vocabulary; the binary mapping is decided after this audit.
    raise NotImplementedError("TODO[Todos]: implement the semantic label audit")


label_audit = audit_labels(repo_path(CFG.frames_dir), inventory["frame_names"])
assert isinstance(label_audit["unique_values"], set), (
    "unique_values must be a set of integers, not a list or a numpy scalar"
)
assert label_audit["unique_values"], "the audit found no label values; check the mask path"
assert set(label_audit["value_counts"]) == set(inventory["frame_names"]), (
    "every frame needs one entry in value_counts"
)
assert all(
    isinstance(value, int) and count >= 0
    for counts in label_audit["value_counts"].values()
    for value, count in counts.items()
), "label values and counts must be non-negative integers"
print(f"Observed label values: {sorted(label_audit['unique_values'])}")
print(f"Frames with a single label value: {len(label_audit['empty_masks'])}")

In [ ]:
# OWNER: Todos
def summarize_anomaly_ratio(
    value_counts: dict[str, dict[int, int]],
    anomaly_labels: tuple[int, ...],
    ignore_labels: tuple[int, ...] = (),
) -> dict[str, float]:
    """Compute per-frame anomalous-pixel ratios after the binary mapping is set.

    Inputs: per-frame value counts, audited anomaly labels and ignored labels.
    Outputs: frame -> ratio in [0, 1], using all non-ignored pixels as denominator.
    Contract: one ratio per frame; no NaN; ratios stay in [0, 1].
    Pitfall: pooling all pixels before dividing hides frames with almost no
    anomaly. Report the per-frame distribution, including the smallest frames.
    """
    # Hint: sum the counts whose key is in anomaly_labels, then divide by the sum
    # of all counts for that frame. Treat an all-background frame as ratio 0.0
    # only after the empty-mask rule is written in 1.4.
    raise NotImplementedError("TODO[Todos]: implement per-frame anomaly coverage")


assert CFG.anomaly_label is not None, (
    "Set CFG.anomaly_label from the audit before computing coverage; do not guess it"
)
ignore_labels = CFG.ignore_labels or ()
anomaly_ratio = summarize_anomaly_ratio(
    label_audit["value_counts"], (CFG.anomaly_label,), ignore_labels
)
assert set(anomaly_ratio) == set(inventory["frame_names"]), (
    "coverage must be reported for every frame, not a hand-picked subset"
)
assert all(0.0 <= ratio <= 1.0 for ratio in anomaly_ratio.values()), (
    "anomaly ratios must be fractions in [0, 1]"
)
assert CFG.anomaly_label not in ignore_labels, "the anomaly value cannot also be an ignored value"

In [ ]:
# OWNER: Todos
def run_quality_checks(frames_dir: Path, frame_names: list[str]) -> dict[str, Any]:
    """Check pairing, size parity, decodability and duplicate content.

    Inputs: frames directory and the ordered frame names.
    Outputs: `size_mismatches`, `duplicate_groups`, `unreadable_masks` and
    `unpaired`; each is a list, and `duplicate_groups` is a list of lists.
    Contract: no size mismatch, unreadable mask or unpaired directory survives.
    Duplicates are reported, then accepted or removed by a written decision,
    because two names for one scene would leak across a split.
    Pitfall: hashing the compressed file misses a re-encoded duplicate; hash the
    decoded pixels if the same photograph may appear under two names.
    """
    # Hint: compare Image.size for frame and labels_semantic.png; hash
    # Image.tobytes() (or a decoded array) with hashlib.sha256. Group equal hashes.
    raise NotImplementedError("TODO[Todos]: implement the dataset quality checks")


quality = run_quality_checks(repo_path(CFG.frames_dir), inventory["frame_names"])
assert not quality["size_mismatches"], (
    f"frame and mask sizes differ: {quality['size_mismatches'][:5]}"
)
assert not quality["unreadable_masks"], (
    f"unreadable semantic masks: {quality['unreadable_masks'][:5]}"
)
assert not quality["unpaired"], f"anchor/pair mismatch: {quality['unpaired'][:5]}"
print(f"Duplicate groups to review: {len(quality['duplicate_groups'])}")

In [ ]:
# OWNER: Todos
def plot_frame_grid(
    frames_dir: Path, frame_names: list[str], anomalies: dict[str, float], n: int = 12
) -> Any:
    """Plot a small image/mask grid for visual inspection.

    Inputs: frames directory, ordered names, the coverage mapping and a display
    limit. Outputs: a Matplotlib `Figure` with paired image and mask axes.
    Contract: axes are even in number, one image and one mask per selected frame,
    masks use nearest-neighbour display, and no title claims a result.
    Pitfall: a pretty grid is not evidence of coverage; label every panel with
    the frame name and keep the per-frame ratio visible in the caption.
    """
    # Hint: matplotlib.pyplot.subplots with two columns; Image.open + imshow for
    # the frame, the semantic PNG for the mask, interpolation="nearest" and
    # axis("off"). Import pyplot inside the cell when you implement it.
    raise NotImplementedError("TODO[Todos]: implement the visual sanity grid")


figure = plot_frame_grid(repo_path(CFG.frames_dir), inventory["frame_names"], anomaly_ratio)
assert figure.axes, "the grid must contain at least one image/mask pair"
assert len(figure.axes) % 2 == 0, "every image panel needs a matching mask panel"

### 1.3 Limitations

Do not write a conclusion before the audit. Record only what the checked data
supports.

> Write your observation here.

- [ ] [Todos] Name the coverage range, the object-scale range and any duplicate or
  unreadable frame that survived the checks.
- [ ] [Todos] State what the single annotation class can and cannot support.
- [ ] [Todos] State that the 60 frames are an evaluation set, not a training set.
- [ ] [Todos] Confirm that `data/credits.txt` travels with every published frame.

### 1.4 Binary target and empty-mask rule

Set the three label fields only after the audit. The decision is part of the
protocol, not a default inherited from a tutorial.

| Decision | Value | Evidence |
|---|---|---|
| Background label | TODO[Todos] | TODO[Todos] |
| Anomaly label | TODO[Todos] | TODO[Todos] |
| Ignored / void labels | TODO[Todos] | TODO[Todos] |
| Empty ground-truth mask | TODO[Todos] | TODO[Todos] |
| Image with no anomaly | TODO[Todos] | TODO[Todos] |

> Write the empty-mask scoring rule here. Say whether the frame is excluded,
> scored as a defined edge case, or reported separately, and say how the frame
> count is preserved. Do not let an empty mask become a perfect score by
> accident.

## 2. Data Preparation

**Goal.** Build a leakage-safe, reproducible input pipeline on the normal-road
corpus, with one mask convention shared by every approach.

**Why it matters.** Augmenting before splitting leaks transformed copies into
validation; bilinear mask resize invents labels.

**Feynman check.** The inspector studies normal roads all week. The exam paper
stays in a sealed envelope; no page is photocopied into the study notes.

**Watch out.** The normal-road release, licence, scene list and split hash must be
pinned. A split created by name order is not a split.

**Your task.**
- [ ] [Todos] Pin the manifest and licence, then create train/val/test by scene before augmentation.
- [ ] [Todos] Implement loaders with nearest mask resize, a train-only shuffle and synchronized augmentation.
- [ ] [Todos] Measure imbalance, write the loss justification and the empty-mask rule.
- [ ] [Khaled] Select normal images for the autoencoder and assert that suitable ones exist.

### 2.1 Pinned manifest and scene-grouped split

Load the normal-road manifest with its release, licence and records, then split
by scene before any augmentation.

In [ ]:
# OWNER: Todos
def load_normal_road_manifest(path: Path) -> dict[str, Any]:
    """Load and validate the pinned normal-road dataset manifest.

    Inputs: repository-relative path to the manifest (YAML or JSON).
    Outputs: mapping with `release`, `license`, `root` and `records`; each record
    has `image`, `mask` and `scene_id`.
    Contract: every record names one image, one mask and one scene; the release
    and licence are non-empty.
    Pitfall: an unpinned manifest can point at a newer release and change the
    training set without changing the notebook.
    """
    # Hint: yaml.safe_load or json.load; validate paths stay under `root`; do not
    # download or re-split anything here. Record the manifest hash in the record.
    raise NotImplementedError("TODO[Todos]: implement the normal-road manifest loader")


assert CFG.normal_road_manifest is not None, (
    "Set CFG.normal_road_manifest before preparing data; a run without it is not reproducible"
)
NORMAL_MANIFEST = load_normal_road_manifest(repo_path(CFG.normal_road_manifest))
assert {"release", "license", "root", "records"} <= set(NORMAL_MANIFEST), (
    "manifest must declare release, license, root and records"
)
assert NORMAL_MANIFEST["release"] and NORMAL_MANIFEST["license"], (
    "release and license must be filled; do not inherit them from memory"
)
assert NORMAL_MANIFEST["records"], "the normal-road manifest has no records"
assert all({"image", "mask", "scene_id"} <= set(record) for record in NORMAL_MANIFEST["records"]), (
    "every manifest record needs image, mask and scene_id"
)

In [ ]:
# OWNER: Todos
def make_scene_grouped_split(manifest: dict[str, Any], *, seed: int) -> dict[str, Any]:
    """Create train/val/test scene groups before any augmentation.

    Inputs: the validated manifest and the run seed.
    Outputs: `train`, `val`, `test` (lists of image ids), `scene_groups` (the
    scene ids behind each split), `split_hash` (sha256 of canonical JSON) and
    `frozen=True`.
    Contract: image sets are pairwise disjoint; no scene id appears in two
    splits; the hash changes when any assignment changes.
    Pitfall: splitting images instead of scenes leaks near-duplicate frames from
    one drive into validation and inflates every score.
    """
    # Hint: group records by scene_id first, then split groups with
    # sklearn.model_selection.GroupShuffleSplit or GroupKFold. Hash a sorted,
    # canonical JSON structure, not a Python repr.
    raise NotImplementedError("TODO[Todos]: implement the leakage-safe split")


SPLIT = make_scene_grouped_split(NORMAL_MANIFEST, seed=CFG.seed)
assert {"train", "val", "test", "scene_groups", "split_hash", "frozen"} <= set(SPLIT), (
    "split must expose train, val, test, scene_groups, split_hash and frozen"
)
assert set(SPLIT["train"]).isdisjoint(SPLIT["val"]), "train and val share an image id"
assert set(SPLIT["train"]).isdisjoint(SPLIT["test"]), "train and test share an image id"
assert set(SPLIT["val"]).isdisjoint(SPLIT["test"]), "val and test share an image id"
assert SPLIT["frozen"] is True, "the test split must be frozen before training starts"
assert len(SPLIT["split_hash"]) == 64, "split_hash must be a full sha256 hex digest"

In [ ]:
# OWNER: Todos
def write_split_manifest(split: dict[str, Any], path: Path) -> Path:
    """Freeze the split assignment as a JSON manifest for later runs.

    Inputs: the split mapping and a destination below the repository.
    Outputs: the written path.
    Contract: the file contains the three id lists, the scene groups, the
    `split_hash` and `frozen`; it is the file `CFG.split_manifest` points to.
    Pitfall: a manifest that is not written cannot be cited by an experiment
    record, and a later re-split silently invalidates the record.
    """
    # Hint: json.dumps(sort_keys=True, indent=2) + newline; create parent dirs;
    # keep repository-relative references in the stored document.
    raise NotImplementedError("TODO[Todos]: write the frozen split manifest")


assert CFG.split_manifest is not None, (
    "Set CFG.split_manifest to the path where the frozen split will be stored"
)
SPLIT_PATH = write_split_manifest(SPLIT, repo_path(CFG.split_manifest))
assert SPLIT_PATH.is_file(), "write_split_manifest must create the file it returns"

### 2.2 Input pipeline and augmentation

Build the three datasets from the frozen split. Mask resize is nearest-neighbour
and augmentation is applied to training batches only.

In [ ]:
# OWNER: Todos
def make_datasets(cfg: Config) -> tuple[tf.data.Dataset, tf.data.Dataset, tf.data.Dataset]:
    """Build the batched train, validation and test datasets.

    Inputs: the configuration, including `image_size`, `batch_size`, the pinned
    manifest and the frozen split manifest.
    Outputs: three `tf.data.Dataset` objects. Each element is
    `(image, mask)`: image float32 `(B, H, W, 3)` in [0, 1]; mask int32
    `(B, H, W, 1)` in {0, 1}, with ignored labels removed by explicit mapping.
    Contract: train shuffles with `cfg.seed`; val and test do not shuffle; masks
    are resized with nearest-neighbour; no augmentation is applied here.
    Pitfall: bilinear mask resize invents fractional labels, and a shuffle on
    validation makes the reported score depend on iteration order.
    """
    # Hint: tf.data.Dataset.from_tensor_slices over manifest records; decode with
    # tf.io.decode_image, resize with tf.image.resize (nearest for masks), then
    # batch and prefetch(tf.data.AUTOTUNE). Shuffle only train.
    raise NotImplementedError("TODO[Todos]: implement the tf.data input pipeline")


assert CFG.image_size is not None and CFG.batch_size is not None, (
    "Set CFG.image_size and CFG.batch_size before building datasets"
)
assert CFG.background_label is not None and CFG.anomaly_label is not None, (
    "Set CFG.background_label and CFG.anomaly_label from the Section 1 audit"
)
assert CFG.ignore_labels is not None, (
    "Set CFG.ignore_labels from the audit; use () when no value is ignored"
)
TRAIN_DS, VAL_DS, TEST_DS = make_datasets(CFG)
for split_name, dataset in (("train", TRAIN_DS), ("val", VAL_DS), ("test", TEST_DS)):
    image_batch, mask_batch = next(iter(dataset))
    assert image_batch.dtype == tf.float32, f"{split_name}: images must be float32"
    assert tuple(image_batch.shape[1:]) == (*CFG.image_size, 3), (
        f"{split_name}: image shape must be (B, H, W, 3) with CFG.image_size"
    )
    assert mask_batch.dtype == tf.int32, f"{split_name}: masks must be int32"
    assert tuple(mask_batch.shape[1:]) == (*CFG.image_size, 1), (
        f"{split_name}: mask shape must be (B, H, W, 1) with CFG.image_size"
    )
    assert float(tf.reduce_min(image_batch)) >= 0.0, f"{split_name}: image below 0"
    assert float(tf.reduce_max(image_batch)) <= 1.0, f"{split_name}: image above 1"
    assert int(tf.reduce_min(mask_batch)) >= 0, f"{split_name}: negative mask value"
    assert int(tf.reduce_max(mask_batch)) <= 1, f"{split_name}: mask is not binary"
assert SPLIT["frozen"] is True, "test must stay frozen while datasets are built"

In [ ]:
# OWNER: Todos
def augment_train_only(image: tf.Tensor, mask: tf.Tensor, seed: int) -> tuple[tf.Tensor, tf.Tensor]:
    """Apply one synchronized random geometry to a training image and its mask.

    Inputs: one float32 `(H, W, 3)` image, one int32 `(H, W, 1)` mask and a seed
    supplied per epoch by the caller.
    Outputs: transformed image and mask with unchanged shapes and dtypes.
    Contract: the same geometric transform is applied to both tensors; the mask
    stays binary; the same seed reproduces the same result.
    Pitfall: independent flips or crops on image and mask teach the model a
    boundary that does not exist. Never call this on validation or test.
    """
    # Hint: tf.image.stateless_random_flip_left_right and
    # stateless_random_crop take an explicit seed tuple; pass the same seed to
    # both tensors. Use nearest-neighbour if the transform resizes the mask.
    raise NotImplementedError("TODO[Todos]: implement synchronized train-only augmentation")


sample_image, sample_mask = next(iter(TRAIN_DS))
sample_image, sample_mask = sample_image[0], sample_mask[0]
aug_image, aug_mask = augment_train_only(sample_image, sample_mask, seed=CFG.seed)
assert aug_image.shape == sample_image.shape, "augmentation must preserve the image shape"
assert aug_mask.shape == sample_mask.shape, "augmentation must preserve the mask shape"
assert aug_image.dtype == sample_image.dtype, "augmentation must preserve the image dtype"
assert aug_mask.dtype == sample_mask.dtype, "augmentation must preserve the mask dtype"
assert set(np.unique(aug_mask.numpy()).tolist()) <= {0, 1}, (
    "augmentation must not invent mask values"
)
_, repeated_mask = augment_train_only(sample_image, sample_mask, seed=CFG.seed)
assert np.array_equal(aug_mask.numpy(), repeated_mask.numpy()), (
    "the same seed must reproduce the same mask transform"
)

### 2.3 Class balance and loss justification

Measure the pixel balance of the training split before choosing a loss. Use the
measured ratio, not an assumption about it.

In [ ]:
# OWNER: Todos
def summarize_class_imbalance(dataset: tf.data.Dataset) -> dict[str, float]:
    """Measure the pixel balance of one split.

    Inputs: a batched `(image, mask)` dataset; use the training split, never the
    frozen test split for this measurement.
    Outputs: `background_pixels`, `anomaly_pixels`, `ignored_pixels`,
    `background_ratio` and `anomaly_ratio`.
    Contract: counts are non-negative; the two ratios sum to 1 after ignored
    pixels are excluded; no NaN.
    Pitfall: a pooled ratio hides frames with almost no anomaly. Keep the
    per-frame coverage from Section 1 beside this number.
    """
    # Hint: iterate batches and np.bincount the integer masks; accumulate counts
    # before dividing. Do not average per-batch ratios.
    raise NotImplementedError("TODO[Todos]: measure class imbalance")


imbalance = summarize_class_imbalance(TRAIN_DS)
assert imbalance["background_pixels"] >= 0 and imbalance["anomaly_pixels"] >= 0, (
    "pixel counts cannot be negative"
)
assert abs(imbalance["background_ratio"] + imbalance["anomaly_ratio"] - 1.0) < 1e-6, (
    "background and anomaly ratios must sum to 1 after ignored pixels are excluded"
)
assert 0.0 <= imbalance["anomaly_ratio"] <= 1.0, "anomaly ratio must be a fraction"

**Loss justification.**

> Write your observation here: which training loss does the measured balance
> justify, and what would go wrong with an unweighted pixel loss? Name the
> measured imbalance and the exact loss you will implement in 3.5.

- [ ] [Todos] Tie the loss choice to the measured ratio, not to a tutorial default.
- [ ] [Todos] State whether class weights, sampling or a region-based term is used.
- [ ] [Todos] State how ignored labels are excluded from the loss.
- [ ] [Todos] Confirm that the choice was made before scoring the test split.

### 2.4 Autoencoder normal-image selection

Select normal-road training images only. The autoencoder never sees an anomalous
target and never sees validation or test images.

In [ ]:
# OWNER: Khaled
def select_normal_images(manifest: dict[str, Any], split: dict[str, Any]) -> list[str]:
    """Select normal-road training images suitable for autoencoder training.

    Inputs: the validated manifest and the frozen split.
    Outputs: image ids drawn only from `split["train"]`, with a readable image,
    a decodable mask and at least one labelled road pixel.
    Contract: the returned ids are a subset of train and disjoint from val/test;
    the list is non-empty. If it is empty, stop and record that validated
    external normal-road data is required before training.
    Pitfall: the normal-road corpus is normal by construction. Filter on data
    quality and scene coverage, not on a guessed anomaly label.
    """
    # Hint: join manifest records to split["train"] by image id; open the mask and
    # reject records with no labelled pixels. Keep the selection reproducible and
    # store the id list beside the split manifest.
    raise NotImplementedError("TODO[Khaled]: select suitable normal training images")


if CFG.approach == "autoencoder":
    NORMAL_IMAGES = select_normal_images(NORMAL_MANIFEST, SPLIT)
    assert NORMAL_IMAGES, (
        "No suitable normal images: document the need for validated external normal-road data"
    )
    assert set(NORMAL_IMAGES) <= set(SPLIT["train"]), (
        "autoencoder selection must stay inside the training split"
    )
    assert set(NORMAL_IMAGES).isdisjoint(SPLIT["val"]), (
        "autoencoder selection leaked a validation image"
    )
    assert set(NORMAL_IMAGES).isdisjoint(SPLIT["test"]), "autoencoder selection leaked a test image"

In [ ]:
# OWNER: Khaled
def make_autoencoder_dataset(image_ids: list[str], cfg: Config) -> tf.data.Dataset:
    """Build the normal-image dataset used to train the autoencoder.

    Inputs: selected image ids and the configuration.
    Outputs: a `tf.data.Dataset` whose elements are float32 `(B, H, W, 3)` images
    in [0, 1]; no masks and no anomaly labels.
    Contract: batches have the configured spatial size; no target image comes
    from val or test; preprocessing matches the reconstruction call in 3.3.
    Pitfall: duplicating the resize/normalize logic here creates a train/inference
    mismatch that looks like a model failure.
    """
    # Hint: reuse the decode/resize helpers from `make_datasets`; call `.map`
    # with the same image normalization and batch/prefetch settings. Do not
    # implement a second preprocessing path.
    raise NotImplementedError("TODO[Khaled]: build the normal-image autoencoder dataset")


if CFG.approach == "autoencoder":
    assert CFG.image_size is not None, "Set CFG.image_size before the autoencoder dataset"
    AE_TRAIN_DS = make_autoencoder_dataset(NORMAL_IMAGES, CFG)
    ae_image_batch = next(iter(AE_TRAIN_DS))
    assert ae_image_batch.dtype == tf.float32, "autoencoder images must be float32"
    assert tuple(ae_image_batch.shape[1:]) == (*CFG.image_size, 3), (
        "autoencoder images must match CFG.image_size and have three channels"
    )
    assert float(tf.reduce_min(ae_image_batch)) >= 0.0, "autoencoder image below 0"
    assert float(tf.reduce_max(ae_image_batch)) <= 1.0, "autoencoder image above 1"

## 3. Model Selection and Training

**Goal.** Build and train one active approach at a time behind a shared
score-map contract.

**Why it matters.** Comparing architectures is only fair when the split, mask
convention and evaluation protocol are identical. A second preprocessing path
turns a comparison of tools into a comparison of pipelines.

**Feynman check.** Three inspectors get the same case file and rule; only their
tools differ. One uses a magnifying glass, one studies only normal roads, one
passes notes between desks.

**Watch out.** Only the active `CFG.approach` runs. Do not train on RoadAnomaly,
and do not select a checkpoint or threshold on its frames.

**Your task.**
- [ ] [Ines · Emna] Build, configure and train DeepLabV3+.
- [ ] [Khaled] Build and train the autoencoder on normal images only.
- [ ] [Hejer] Train the CNN encoder-decoder baseline; [Walid] build and train U-Net.
- [ ] [Todos] Implement the shared loss, callbacks, record and curves.

In [ ]:
# OWNER: Todos
# Runtime handles filled by the active model section; never fit on RoadAnomaly.
ACTIVE_MODEL: keras.Model | None = None
ACTIVE_HISTORY: keras.callbacks.History | None = None
MODELS: dict[str, keras.Model] = {}

### 3.1 Approach table

**Goal.** Freeze the closed list of approaches before any code is written.

**Why it matters.** A fourth model or a renamed score quietly invalidates the
comparison table and the ownership map.

**Feynman check.** The shift supervisor lists the three tools allowed on the
case, so nobody signs out a different one halfway through.

**Watch out.** The CNN encoder-decoder is the U-Net baseline (`skip_connections=False`), not
a fourth approach. XAI is not a network.

**Your task.**
- [ ] [Todos] Use only the `APPROACH` strings below in code and records.
- [ ] [Ines · Emna] Confirm the DeepLabV3+ training signal is supervised masks.
- [ ] [Khaled · Adrian] Confirm the autoencoder sees normal images only.
- [ ] [Hejer · Walid] Confirm U-Net and its baseline share one mask convention.

| `APPROACH` | Model | Training signal | Owners |
|---|---|---|---|
| `"deeplabv3plus"` | DeepLabV3+, pretrained encoder, ASPP, decoder; frozen or fine-tuned | Supervised masks | Ines · Emna |
| `"autoencoder"` | Convolutional autoencoder; error map thresholded into a mask | Normal images only | Khaled · Adrian |
| `"unet"` | U-Net with skips, plus CNN encoder-decoder baseline (`skip_connections=False`) | Supervised masks | Hejer · Walid |

### 3.2 DeepLabV3+ — Ines · Emna

**Goal.** Build DeepLabV3+ with a pretrained encoder, ASPP and a decoder, then
train it on normal-road supervised masks.

**Why it matters.** A frozen encoder and a fine-tuned encoder answer different
questions. A number without the regime and weight source cannot be compared.

**Feynman check.** The inspector uses a magnifying glass at several zoom levels
(ASPP) and a sketch pad (decoder). Pretrained weights are experience from another
depot, not from this road.

**Watch out.** Encoder weights carry their own licence and preprocessing. A
mismatch between training and inference preprocessing shifts every score.

**Your task.**
- [ ] [Ines] Implement the builder and assert the output shape.
- [ ] [Emna] Record the weight source, licence and preprocessing.
- [ ] [Emna] Choose frozen or fine-tuned and record the reason.
- [ ] [Ines · Emna] Implement Seg-Grad-CAM in 4.3.

#### 3.2.1 Pretrained encoder record — Emna

Fill every field before the first training run. Do not estimate the parameter
count; measure it from the built model.

| Field | Value |
|---|---|
| Encoder family and variant | TODO[Emna] |
| Weight source and version | TODO[Emna] |
| Weight licence | TODO[Emna] |
| Input preprocessing (scale, mean, colour order) | TODO[Emna] |
| Frozen or fine-tuned | TODO[Emna] |
| Trainable parameter count | TODO[Emna] — measured from `model.count_params()` |
| ASPP dilation rates and justification | TODO[Emna] |

> Write your observation here: why this encoder and this frozen/fine-tuned
> regime fit the normal-road corpus and the available compute.

In [ ]:
# OWNER: Ines
def build_deeplabv3plus(freeze_encoder: bool = True) -> keras.Model:
    """Build DeepLabV3+ with a pretrained encoder, ASPP and a decoder.

    Inputs: `freeze_encoder` — True freezes pretrained weights, False fine-tunes.
    Uses `CFG.image_size`; set it before calling.
    Outputs: `keras.Model` mapping float32 `(B, H, W, 3)` to per-pixel logits
    `(B, H, W, C)` at the same spatial size. `C` follows the verified label map.
    Contract: rank-4 output, spatial size preserved, finite logits.
    Pitfall: freeze before `compile`; unfreezing later requires a recompile.
    Record which regime produced the checkpoint.
    """
    # Hint: keras.applications for the encoder; Conv2D with dilation_rate for
    # ASPP branches; UpSampling2D + Concatenate for the decoder. Do not train the
    # encoder on RoadAnomaly.
    raise NotImplementedError("TODO[Ines]: implement the DeepLabV3+ builder")


if CFG.approach == "deeplabv3plus":
    assert CFG.image_size is not None and CFG.freeze_encoder is not None, (
        "Set CFG.image_size and CFG.freeze_encoder before building DeepLabV3+"
    )
    DEEPLAB_MODEL = build_deeplabv3plus(freeze_encoder=CFG.freeze_encoder)
    assert len(DEEPLAB_MODEL.output.shape) == 4, (
        "DeepLabV3+ must return a rank-4 tensor (batch, height, width, channels)"
    )
    assert tuple(DEEPLAB_MODEL.output.shape[1:3]) == CFG.image_size, (
        "DeepLabV3+ output must keep the input spatial size"
    )
    MODELS["deeplabv3plus"] = DEEPLAB_MODEL

In [ ]:
# OWNER: Emna
def train_deeplabv3plus(
    model: keras.Model,
    cfg: Config,
    train_ds: tf.data.Dataset,
    val_ds: tf.data.Dataset,
    *,
    loss: keras.losses.Loss,
    metrics: list[keras.metrics.Metric],
    callbacks: list[keras.callbacks.Callback],
) -> keras.callbacks.History:
    """Train DeepLabV3+ and return the Keras history.

    Inputs: the built model, the configuration, the training split, the
    validation split, and the shared loss, monitoring metrics and callbacks from
    3.5. The test split is not an input.
    Outputs: `keras.callbacks.History` with `loss` and `val_loss` recorded.
    Contract: at least one epoch ran; the number of epochs is at most
    `cfg.epochs`; validation monitors `val_loss` (or a declared validation
    metric), never a test value.
    Pitfall: early stopping must restore or reload the best validation weights,
    otherwise the returned model is the last epoch, not the best one.
    """
    # Hint: model.compile(optimizer=keras.optimizers.Adam(learning_rate=...),
    # loss=loss, metrics=metrics), then model.fit(..., validation_data=val_ds,
    # callbacks=callbacks). Store provenance in the record, not only in a print.
    raise NotImplementedError("TODO[Emna]: implement DeepLabV3+ training")

### 3.3 Autoencoder — Khaled

**Goal.** Build a convolutional autoencoder, reconstruct normal-road images and
train it only on normal-road images.

**Why it matters.** Reconstruction error is the anomaly signal. If the model is
trained on anomalies, it learns to reconstruct them and the signal collapses.

**Feynman check.** This inspector has only seen normal roads. Anything unfamiliar
comes back from memory distorted; the distortion is the clue.

**Watch out.** A model that copies its input too well reconstructs large, smooth
anomalies as normal. Do not tune that away on the test split.

**Your task.**
- [ ] [Khaled] Implement the builder and assert input/output shapes.
- [ ] [Khaled] Implement `reconstruct` and document the output range.
- [ ] [Khaled] Train on `AE_TRAIN_DS` only.
- [ ] [Adrian] Turn reconstructions into error maps and a validation threshold in 4.2.

In [ ]:
# OWNER: Khaled
def build_autoencoder() -> keras.Model:
    """Build the convolutional autoencoder.

    Inputs: none; reads `CFG.image_size`. Set it before calling.
    Outputs: `keras.Model` mapping float32 `(B, H, W, 3)` to a reconstruction of
    the same shape.
    Contract: output spatial size equals input spatial size; the encoder has a
    smaller spatial bottleneck than the input; no skip connection copies detail
    across the bottleneck.
    Pitfall: an identity-like model reconstructs anomalies too well, which looks
    like a strong detector until the error maps are inspected.
    """
    # Hint: Conv2D + MaxPooling2D encoder, Conv2DTranspose/UpSampling2D decoder.
    # Print model.summary() and confirm the bottleneck before training.
    raise NotImplementedError("TODO[Khaled]: implement the autoencoder builder")


if CFG.approach == "autoencoder":
    assert CFG.image_size is not None, "Set CFG.image_size before building the autoencoder"
    AE_MODEL = build_autoencoder()
    AE_SAMPLE = next(iter(TRAIN_DS))[0]
    AE_SAMPLE_RECON = AE_MODEL(AE_SAMPLE, training=False)
    assert tuple(AE_SAMPLE_RECON.shape[1:]) == tuple(AE_SAMPLE.shape[1:]), (
        "the autoencoder must reconstruct the input spatial size and channels"
    )
    assert bool(np.isfinite(AE_SAMPLE_RECON.numpy()).all()), (
        "the untrained autoencoder produced non-finite values"
    )
    MODELS["autoencoder"] = AE_MODEL

In [ ]:
# OWNER: Khaled
def reconstruct(model: keras.Model, images: np.ndarray) -> np.ndarray:
    """Reconstruct a batch of images for the error-map stage.

    Inputs: the autoencoder and float32 images `(N, H, W, 3)` in [0, 1].
    Outputs: float32 reconstructions `(N, H, W, 3)` with the same spatial size.
    Contract: shape equals the input shape; all values are finite; the output
    range follows the decoder activation and is recorded in the experiment file.
    Pitfall: do not clip silently. Clipping hides saturation and changes the
    error map that 4.2 thresholds.
    """
    # Hint: model.predict(images, verbose=0) for a NumPy path, or model(images,
    # training=False) for a tensor path. Keep the same preprocessing as training.
    raise NotImplementedError("TODO[Khaled]: implement the reconstruction helper")


if CFG.approach == "autoencoder":
    AE_INPUT = next(iter(TRAIN_DS))[0].numpy()
    AE_RECON = reconstruct(AE_MODEL, AE_INPUT)
    assert AE_RECON.shape == AE_INPUT.shape, "reconstruction shape must equal input shape"
    assert AE_RECON.dtype == np.float32, "reconstruction must be float32 for error maps"
    assert np.isfinite(AE_RECON).all(), "reconstruction contains NaN or infinity"

In [ ]:
# OWNER: Khaled
def train_autoencoder(
    model: keras.Model,
    cfg: Config,
    normal_ds: tf.data.Dataset,
    *,
    loss: keras.losses.Loss,
    metrics: list[keras.metrics.Metric],
    callbacks: list[keras.callbacks.Callback],
) -> keras.callbacks.History:
    """Train the autoencoder on normal-road images only.

    Inputs: the built model, the configuration, `AE_TRAIN_DS` (images only), and
    the shared loss, monitoring metrics and callbacks from 3.5.
    Outputs: `keras.callbacks.History` with `loss` and `val_loss`.
    Contract: the target is the input image; no anomaly mask or benchmark frame
    is used; early stopping monitors validation reconstruction loss.
    Pitfall: passing `(image, mask)` pairs to a reconstruction loss silently makes
    the mask the target; keep the image-only dataset.
    """
    # Hint: compile with the reconstruction loss, then fit on AE_TRAIN_DS with
    # validation_data=(validation images, validation images); use callbacks from
    # 3.5. Keep masks out of the autoencoder path.
    raise NotImplementedError("TODO[Khaled]: implement autoencoder training")

### 3.4 CNN baseline and U-Net — Hejer · Walid

**Goal.** Train the CNN encoder-decoder baseline first, then U-Net, on the same
splits and mask convention.

**Why it matters.** The baseline isolates what skip connections buy. If depth,
width or augmentation also change, the comparison answers a different question.

**Feynman check.** One inspector works without passing notes between desks; the
other hands a note from the intake desk to the drawing desk. Same case, one
difference.

**Watch out.** The baseline is `build_unet(skip_connections=False)`, not a
smaller architecture. The reported model must be the reloaded best checkpoint,
not the last epoch.

**Your task.**
- [ ] [Hejer] Build and train the baseline with `skip_connections=False`.
- [ ] [Walid] Build U-Net with `skip_connections=True`.
- [ ] [Walid] Configure early stopping and best-model saving on validation.
- [ ] [Hejer · Walid] Report both through the shared Section 4.1 metrics; do not write a second metric.

In [ ]:
# OWNER: Walid
def build_unet(skip_connections: bool = True) -> keras.Model:
    """Build U-Net, or its CNN encoder-decoder baseline when skips are disabled.

    Inputs: `skip_connections` — True adds encoder-to-decoder concatenation;
    False removes only that path. Uses `CFG.image_size`; set it before calling.
    Outputs: `keras.Model` mapping float32 `(B, H, W, 3)` to per-pixel logits
    `(B, H, W, C)` at the same spatial size.
    Contract: rank-4 output and preserved spatial size; the two variants share
    depth, width and decoder structure, differing only by the skip path.
    Pitfall: changing depth or width together with the flag confounds the
    baseline comparison; it is then a different experiment.
    """
    # Hint: Conv2D + MaxPooling2D encoder; Conv2DTranspose + Concatenate decoder.
    # Gate only the concatenation on `skip_connections`; keep every other layer
    # identical between the two variants.
    raise NotImplementedError("TODO[Walid]: implement the shared U-Net builder")


if CFG.approach == "unet":
    assert CFG.image_size is not None, "Set CFG.image_size before building the CNN baseline"
    BASELINE_MODEL = build_unet(skip_connections=False)
    assert len(BASELINE_MODEL.output.shape) == 4, "the CNN baseline must return a rank-4 tensor"
    assert tuple(BASELINE_MODEL.output.shape[1:3]) == CFG.image_size, (
        "the CNN baseline must preserve the input spatial size"
    )

In [ ]:
# OWNER: Hejer
def train_cnn_baseline(
    model: keras.Model,
    cfg: Config,
    train_ds: tf.data.Dataset,
    val_ds: tf.data.Dataset,
    *,
    loss: keras.losses.Loss,
    metrics: list[keras.metrics.Metric],
    callbacks: list[keras.callbacks.Callback],
) -> keras.callbacks.History:
    """Train the CNN encoder-decoder baseline as the first supervised pass.

    Inputs: the baseline model, configuration, train/val splits, and the shared
    loss, monitoring metrics and callbacks from 3.5.
    Outputs: `keras.callbacks.History` with `loss` and `val_loss`.
    Contract: the test split is not an input; early stopping monitors validation;
    the returned history covers at least one epoch.
    Pitfall: the baseline's metrics are not a second metric implementation. Use
    the shared Section 4.1 functions for the reported numbers.
    """
    # Hint: model.compile(optimizer=..., loss=loss, metrics=metrics), then
    # model.fit(train_ds, validation_data=val_ds, callbacks=callbacks).
    # Prepare segmentation tensors exactly as in 2.2; do not add a private path.
    raise NotImplementedError("TODO[Hejer]: implement CNN baseline training")

**Owner:** Hejer. Baseline numbers are reported through the shared functions in
4.1. Do not write Precision, Recall, F1, IoU or Dice again in this cell; a second
implementation is a bug even when it returns the same value.

In [ ]:
# OWNER: Walid
if CFG.approach == "unet":
    assert CFG.image_size is not None, "Set CFG.image_size before building U-Net"
    UNET_MODEL = build_unet(skip_connections=True)
    assert len(UNET_MODEL.output.shape) == 4, "U-Net must return a rank-4 tensor"
    assert tuple(UNET_MODEL.output.shape[1:3]) == CFG.image_size, (
        "U-Net must preserve the input spatial size"
    )
    assert UNET_MODEL.count_params() != BASELINE_MODEL.count_params(), (
        "the skip path should add parameters; if counts match, the flag was ignored"
    )
    MODELS["unet"] = UNET_MODEL

In [ ]:
# OWNER: Walid
def train_unet(
    model: keras.Model,
    cfg: Config,
    train_ds: tf.data.Dataset,
    val_ds: tf.data.Dataset,
    *,
    loss: keras.losses.Loss,
    metrics: list[keras.metrics.Metric],
    callbacks: list[keras.callbacks.Callback],
) -> tuple[keras.Model, keras.callbacks.History]:
    """Train U-Net with early stopping and best-model saving.

    Inputs: the U-Net, configuration, train/val splits, and the shared loss,
    metrics and callbacks from 3.5.
    Outputs: the reloaded best-validation model and its `History`.
    Contract: the returned model is loaded from the best checkpoint, not the last
    epoch; validation monitors `val_loss` or a declared validation metric; the
    test split is never an input.
    Pitfall: a checkpoint that is written but never reloaded is not the model you
    report. `restore_best_weights=True` or an explicit `load_model` is required.
    """
    # Hint: ModelCheckpoint(save_best_only=True) and
    # EarlyStopping(restore_best_weights=True) come from 3.5; reload with
    # keras.models.load_model and return that object with the history.
    raise NotImplementedError("TODO[Walid]: implement U-Net training and best-model reload")

### 3.5 Shared training stubs — Todos

**Goal.** Define the loss, monitoring metrics, callbacks, experiment record and
learning curves once for all three approaches.

**Why it matters.** Duplicated training logic lets one model be compared on a
different rule. The shared objects are the only allowed source.

**Feynman check.** The rulebook and logbook sit on the shift desk. Each inspector
uses a different tool, but everyone writes the same fields.

**Watch out.** A Keras metric named `val_dice` is a training monitor, not the
project Dice. Final values come from 4.1. A callback must never monitor test.

**Your task.**
- [ ] [Todos] Implement the loss that matches each training signal.
- [ ] [Walid] Implement checkpoints and early stopping on validation.
- [ ] [Todos] Fill the experiment record and write it with the implemented writer.
- [ ] [Todos] Write the run YAML under `configs/` and record its path and hash.
- [ ] [Todos] Plot learning curves into the run folder.

In [ ]:
# OWNER: Todos
def build_loss(cfg: Config) -> keras.losses.Loss:
    """Return the training loss for the active approach.

    Inputs: the configuration, including `approach` and the audited label policy.
    Outputs: a `keras.losses.Loss` instance.
    Contract: reconstruction loss for the autoencoder; supervised mask loss for
    U-Net and DeepLabV3+. The loss never receives a RoadAnomaly target.
    Pitfall: an unweighted pixel loss can ignore the anomaly class. Justify the
    choice with the Section 2 imbalance measurement, not a default.
    """
    # Hint: keras.losses.MeanSquaredError for reconstruction; BinaryCrossentropy
    # or CategoricalCrossentropy for masks, with from_logits matching the head.
    # Keep the label mapping identical to 2.2.
    raise NotImplementedError("TODO[Todos]: implement the shared loss builder")


assert CFG.approach is not None, "Set CFG.approach before building the loss"
LOSS = build_loss(CFG)
assert isinstance(LOSS, keras.losses.Loss), "build_loss must return a keras.losses.Loss"

In [ ]:
# OWNER: Todos
def make_monitoring_metrics(cfg: Config) -> list[keras.metrics.Metric]:
    """Return training-time monitoring metrics for the active approach.

    Inputs: the configuration.
    Outputs: a non-empty list of Keras metric instances.
    Contract: names are prefixed `train_` or used only in `model.fit`; these are
    monitors, not the reported project metrics. The reported values come from 4.1.
    Pitfall: a Keras `MeanIoU` over a different class set is not the project IoU;
    do not copy its value into the results table.
    """
    # Hint: keras.metrics.Precision, Recall, MeanIoU or BinaryAccuracy. Match the
    # mask encoding used by the loss and by 4.1; no test data in `fit`.
    raise NotImplementedError("TODO[Todos]: implement monitoring metrics")


MONITORING_METRICS = make_monitoring_metrics(CFG)
assert MONITORING_METRICS, "at least one monitoring metric is required"
assert all(isinstance(metric, keras.metrics.Metric) for metric in MONITORING_METRICS), (
    "every monitoring metric must be a keras.metrics.Metric"
)

In [ ]:
# OWNER: Walid
def make_callbacks(cfg: Config, run_dir: Path, tag: str) -> list[keras.callbacks.Callback]:
    """Build checkpointing, early stopping and logging callbacks.

    Inputs: configuration, the run folder and a short tag that keeps baseline and
    U-Net files apart.
    Outputs: non-empty list of Keras callbacks.
    Contract: includes `ModelCheckpoint` under `run_dir/checkpoints/` and
    `EarlyStopping`; both monitor validation, never test. `CSVLogger` writes under
    `run_dir/logs/`; `ReduceLROnPlateau` is optional.
    Pitfall: `restore_best_weights` must be explicit, and two models sharing one
    checkpoint filename overwrite each other.
    """
    # Hint: keras.callbacks.ModelCheckpoint(save_best_only=True),
    # EarlyStopping(restore_best_weights=True, patience=...), CSVLogger and
    # optionally ReduceLROnPlateau. Use `tag` in every filename.
    raise NotImplementedError("TODO[Walid]: implement the shared callbacks")

In [ ]:
# OWNER: Todos
EXPERIMENT_RECORD_FIELDS = (
    "experiment_id",
    "crisp_dm_phase",
    "owner",
    "approach",
    "git_commit",
    "config_path",
    "config_hash",
    "command",
    "seed",
    "python",
    "tensorflow",
    "hardware",
    "normal_road_manifest",
    "split_hash",
    "roadanomaly_commit",
    "threshold",
    "threshold_source",
    "metrics",
    "xai",
    "notes",
)

# Empty values on purpose: the team fills them from a real run. A record with a
# None field is incomplete and must not be published.
EXPERIMENT_RECORD: dict[str, Any] = dict.fromkeys(EXPERIMENT_RECORD_FIELDS)
assert set(EXPERIMENT_RECORD) == set(EXPERIMENT_RECORD_FIELDS), (
    "the record template must expose every field exactly once"
)

In [ ]:
# OWNER: Todos
def write_experiment_record(record: dict[str, Any], destination: Path) -> Path:
    """Write one experiment record as sorted, human-readable JSON.

    Inputs: a record containing every `EXPERIMENT_RECORD_FIELDS` key and a
    destination below the repository.
    Outputs: the written path.
    Contract: missing fields raise before writing; JSON is sorted and indented;
    the file ends with a newline. No metric value is invented here.
    Pitfall: writing a record with `None` provenance makes an unreproducible run
    look documented. The caller checks completeness first.
    """
    missing = [name for name in EXPERIMENT_RECORD_FIELDS if name not in record]
    if missing:
        raise KeyError(f"experiment record is missing fields: {missing}")
    destination = destination.resolve()
    if REPO_ROOT not in destination.parents:
        raise ValueError(f"record destination escapes the repository: {destination}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    payload = dict(record)
    payload["written_at_utc"] = datetime.now(UTC).isoformat()
    destination.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return destination


RECORD_PATH: Path | None = None
PUBLISHED_RECORD_PATH: Path | None = None
if RUN_DIR is not None and all(value is not None for value in EXPERIMENT_RECORD.values()):
    RECORD_PATH = write_experiment_record(
        EXPERIMENT_RECORD, RUN_DIR / "records" / "experiment.json"
    )
    PUBLISHED_RECORD_PATH = write_experiment_record(
        EXPERIMENT_RECORD,
        repo_path(CFG.experiments_dir) / f"{RUN_DIR.name}.json",
    )
assert RECORD_PATH is None or RECORD_PATH.is_file(), (
    "write_experiment_record must create the run record it returns"
)
assert PUBLISHED_RECORD_PATH is None or PUBLISHED_RECORD_PATH.is_file(), (
    "the published record under docs/experiments/ must exist"
)

In [ ]:
# OWNER: Todos
assert CFG.approach is not None, "Set CFG.approach before running a trainer"
assert RUN_DIR is not None, "Create RUN_DIR before running a trainer"
assert CFG.epochs is not None, "Set CFG.epochs before running a trainer"
histories: list[tuple[str, keras.callbacks.History]] = []

if CFG.approach == "deeplabv3plus":
    CALLBACKS = make_callbacks(CFG, RUN_DIR, "deeplabv3plus")
    DEEPLAB_HISTORY = train_deeplabv3plus(
        DEEPLAB_MODEL,
        CFG,
        TRAIN_DS,
        VAL_DS,
        loss=LOSS,
        metrics=MONITORING_METRICS,
        callbacks=CALLBACKS,
    )
    histories.append(("deeplabv3plus", DEEPLAB_HISTORY))
    ACTIVE_MODEL, ACTIVE_HISTORY = DEEPLAB_MODEL, DEEPLAB_HISTORY

In [ ]:
# OWNER: Todos
if CFG.approach == "autoencoder":
    CALLBACKS = make_callbacks(CFG, RUN_DIR, "autoencoder")
    AE_HISTORY = train_autoencoder(
        AE_MODEL,
        CFG,
        AE_TRAIN_DS,
        loss=LOSS,
        metrics=MONITORING_METRICS,
        callbacks=CALLBACKS,
    )
    histories.append(("autoencoder", AE_HISTORY))
    ACTIVE_MODEL, ACTIVE_HISTORY = AE_MODEL, AE_HISTORY

In [ ]:
# OWNER: Todos
if CFG.approach == "unet":
    BASELINE_CALLBACKS = make_callbacks(CFG, RUN_DIR, "unet_baseline")
    BASELINE_HISTORY = train_cnn_baseline(
        BASELINE_MODEL,
        CFG,
        TRAIN_DS,
        VAL_DS,
        loss=LOSS,
        metrics=MONITORING_METRICS,
        callbacks=BASELINE_CALLBACKS,
    )
    UNET_CALLBACKS = make_callbacks(CFG, RUN_DIR, "unet")
    UNET_MODEL, UNET_HISTORY = train_unet(
        UNET_MODEL,
        CFG,
        TRAIN_DS,
        VAL_DS,
        loss=LOSS,
        metrics=MONITORING_METRICS,
        callbacks=UNET_CALLBACKS,
    )
    histories.extend([("cnn_baseline", BASELINE_HISTORY), ("unet", UNET_HISTORY)])
    ACTIVE_MODEL, ACTIVE_HISTORY = UNET_MODEL, UNET_HISTORY

In [ ]:
# OWNER: Todos
assert histories, "the active approach must produce one training history"
for run_name, history in histories:
    assert {"loss", "val_loss"} <= set(history.history), (
        f"{run_name}: history must contain loss and val_loss"
    )
    assert 0 < len(history.history["loss"]) <= CFG.epochs, (
        f"{run_name}: run at least one epoch and never exceed CFG.epochs"
    )

In [ ]:
# OWNER: Todos
def plot_learning_curves(history: keras.callbacks.History, run_dir: Path) -> Path:
    """Plot training and validation loss for one run.

    Inputs: the Keras history and the run folder.
    Outputs: path to a PNG under `run_dir/figures/`.
    Contract: the figure has one loss panel, axes are labelled, and the file is
    saved before returning. No result claim is placed in the title.
    Pitfall: a curve with a rising validation loss is evidence for early stopping,
    not evidence for the benchmark; keep the two arguments separate.
    """
    # Hint: matplotlib.pyplot with history.history["loss"] and ["val_loss"];
    # savefig under run_dir/figures, close the figure, return the path.
    raise NotImplementedError("TODO[Todos]: implement learning-curve plotting")


assert ACTIVE_HISTORY is not None and RUN_DIR is not None, (
    "Run the active trainer before plotting learning curves"
)
CURVE_PATH = plot_learning_curves(ACTIVE_HISTORY, RUN_DIR)
assert CURVE_PATH.is_file() and CURVE_PATH.suffix == ".png", (
    "plot_learning_curves must save a PNG and return its path"
)

## 4. Evaluation and XAI

**Goal.** Score the active model once on the frozen test split, then explain its
errors without turning an explanation into a selection signal.

**Why it matters.** A test split reused for tuning becomes a validation split. A
heatmap that is never checked against the mask can look convincing and still be
unfaithful.

**Feynman check.** The sealed exam is opened once. The inspector then circles the
mistakes on a contact sheet and writes why, but the mark is not changed by how
good the explanation looks.

**Watch out.** The threshold comes from validation only. F1 equals Dice for a
binary mask; do not report them as independent evidence. XAI never promotes a
model.

**Your task.**
- [ ] [Todos] Implement the shared binary metrics and the empty-mask rule.
- [ ] [Adrian] Implement error maps, validation threshold, post-processing, AUROC and AUPRO.
- [ ] [Ines · Emna] Implement DeepLabV3+ Boundary IoU, conditional mIoU and Seg-Grad-CAM.
- [ ] [Hejer · Walid] Implement HD95, Seg-Grad-CAM, Integrated Gradients and the CNN/U-Net comparison.
- [ ] [Khaled · Adrian] Build the error-map panel and write its interpretation.
- [ ] [Todos] Fill the comparison table only from experiment records and write the selection rationale.

### 4.1 Evaluation protocol — Todos

**Goal.** Define Precision, Recall, F1, IoU and Dice once for every approach.

**Why it matters.** Two metric implementations produce two truths. A model must
not be scored by a private copy of Dice.

**Feynman check.** One marking scheme is pinned to the wall. Every inspector's
paper is marked with the same scheme, whether the tool was a magnifying glass or
a memory of normal roads.

**Watch out.** A pixel-pooled mean hides a catastrophic frame. Report the
per-frame distribution, the worst frame and the frame count. Preserve the
empty-mask rule from 1.4.

**Your task.**
- [ ] [Todos] Implement `compute_binary_metrics` and `aggregate_metrics`.
- [ ] [Todos] Assert `F1 == Dice` and `IoU <= Dice` for the same mask.
- [ ] [Todos] State whether the aggregate is a frame mean or a pixel pool.
- [ ] [Todos] Write the empty-mask rule and keep the frame count visible.

In [ ]:
# OWNER: Todos
# Runtime state filled by the approach that runs; the test split is scored in 4.6.
SELECTED_THRESHOLD: float | None = None


def score_map_from_logits(logits: np.ndarray) -> np.ndarray:
    """Convert segmentation logits into a per-pixel anomaly score.

    Inputs: float32 logits `(N, H, W, C)` from U-Net or DeepLabV3+.
    Outputs: float32 anomaly scores `(N, H, W)` in [0, 1], where high means
    "not confidently any trained class".
    Contract: one score per pixel; channel dimension is removed by the fixed
    maximum-softmax rule; finite values in [0, 1].
    Pitfall: average class probabilities before taking the maximum and the score
    means something else. Apply softmax once, then `1 - max`.
    """
    # Hint: tf.nn.softmax or keras.activations.softmax on the last axis, then
    # 1 - np.max(..., axis=-1). Keep the same rule for U-Net and DeepLabV3+.
    raise NotImplementedError("TODO[Todos]: implement the maximum-softmax anomaly score")


if CFG.approach in {"unet", "deeplabv3plus"}:
    assert ACTIVE_MODEL is not None, "Train the segmentation model before scoring"
    VAL_LOGITS = ACTIVE_MODEL(next(iter(VAL_DS))[0], training=False).numpy()
    VAL_SCORES = score_map_from_logits(VAL_LOGITS)
    assert VAL_SCORES.shape == VAL_LOGITS.shape[:3], (
        "the anomaly score must remove the class axis and keep (N, H, W)"
    )
    assert np.isfinite(VAL_SCORES).all(), "anomaly scores contain NaN or infinity"
    assert float(VAL_SCORES.min()) >= 0.0 and float(VAL_SCORES.max()) <= 1.0, (
        "maximum-softmax scores must stay in [0, 1]"
    )

In [ ]:
# OWNER: Todos
def score_dataset(
    model: keras.Model, dataset: tf.data.Dataset, approach: str
) -> tuple[np.ndarray, np.ndarray]:
    """Score one split for the active approach, without thresholding it.

    Inputs: the trained model, a batched dataset and the Scope Lock approach.
    Outputs: `scores` float32 `(N, H, W)` and `masks` int32 `(N, H, W)` in {0, 1}.
    Contract: one score plane per frame and one aligned mask; no threshold is
    applied here; the dataset is never shuffled before this call.
    Pitfall: scoring test frames one at a time and keeping the best-looking ones
    is selection on test. Score the whole split once.
    """
    # Hint: for autoencoder, call compute_error_map; for segmentation, call the
    # model and then score_map_from_logits. Drop the mask channel axis and keep
    # frame order identical to the dataset.
    raise NotImplementedError("TODO[Todos]: implement the shared split scorer")

In [ ]:
# OWNER: Todos
def compute_binary_metrics(pred_mask: np.ndarray, true_mask: np.ndarray) -> dict[str, float]:
    """Compute Precision, Recall, F1, IoU and Dice for one binary mask.

    Inputs: predicted and ground-truth masks `(H, W)`; 1 means anomaly, 0 means
    background. Ignored pixels must already be excluded by the caller.
    Outputs: mapping with the five metric names and finite float values.
    Contract: every value is in [0, 1]; zero denominators return 0.0 rather than
    NaN; for the same mask, F1 equals Dice and IoU is not greater than Dice.
    Pitfall: an empty ground-truth mask is not a perfect frame. Apply the rule
    written in 1.4 and keep the frame in the reported count.
    """
    # Hint: TP/FP/FN from boolean masks; guard each denominator. Compute F1 once
    # and reuse it for Dice instead of writing two independent formulas.
    raise NotImplementedError("TODO[Todos]: implement the binary metric function")

In [ ]:
# OWNER: Todos
def aggregate_metrics(per_image: list[dict[str, float]]) -> dict[str, float]:
    """Aggregate per-frame metrics into one reported row.

    Inputs: one metric mapping per frame, in dataset order.
    Outputs: the same five keys, aggregated across frames.
    Contract: the aggregation rule is named in the experiment record (frame
    mean or pixel pool); no frame is dropped to improve the value.
    Pitfall: a mean of per-frame IoU and a pixel-pooled IoU answer different
    questions. State which one the table uses.
    """
    # Hint: np.mean over the five keys for a frame mean; if you pool, keep the
    # per-frame values too. Never average only the frames that contain anomalies.
    raise NotImplementedError("TODO[Todos]: implement metric aggregation")

#### 4.1.1 Empty-mask rule

> Write the rule here, then use it in `compute_binary_metrics`: is an empty
> ground-truth mask excluded, reported as a defined zero, or scored as a
> separate edge case? State how the frame count and the aggregate are affected.
> Do not let a zero-denominator frame become a silent perfect score.

### 4.2 Autoencoder evaluation — Adrian

**Goal.** Turn reconstructions into per-pixel error maps, choose one threshold on
validation and measure AUROC/AUPRO with their precautions.

**Why it matters.** The error map is the model's anomaly signal; a threshold
chosen on test turns the benchmark into a tuning set.

**Feynman check.** The inspector who only knows normal roads marks how badly each
patch comes back from memory. The suspicion dial is set on practice roads, not on
the sealed exam.

**Watch out.** AUROC can look favourable when very few pixels are anomalous; always
report the anomaly ratio beside it. AUPRO is incomplete without its FPR cap.

**Your task.**
- [ ] [Adrian] Implement error-map aggregation and document the channel rule.
- [ ] [Adrian] Select one threshold on validation only; record the criterion.
- [ ] [Adrian] Implement post-processing and the FP/FN study.
- [ ] [Adrian] Implement pixel AUROC and AUPRO with `CFG.aupro_fpr_cap`.

In [ ]:
# OWNER: Adrian
def compute_error_map(model: keras.Model, images: np.ndarray) -> np.ndarray:
    """Compute a per-pixel reconstruction-error map for a batch.

    Inputs: the autoencoder and float32 images `(N, H, W, 3)` in [0, 1].
    Outputs: float32 non-negative error maps `(N, H, W)`.
    Contract: one score per pixel; finite values; the channel-aggregation rule
    (mean, max or another declared reduction) is written in the experiment
    record; no threshold is applied here.
    Pitfall: using the test split to choose the aggregation is threshold tuning
    by another name. Fix the rule on validation.
    """
    # Hint: reconstruct, take the absolute difference per channel, then reduce
    # the channel axis with the rule you declared. Do not clip the reconstruction
    # before measuring the error.
    raise NotImplementedError("TODO[Adrian]: implement the reconstruction-error map")


if CFG.approach == "autoencoder":
    assert ACTIVE_MODEL is not None, "Train the autoencoder before computing error maps"
    AE_VAL_IMAGES = np.concatenate([images.numpy() for images, _ in VAL_DS], axis=0)
    AE_VAL_MASKS = np.concatenate([masks.numpy() for _, masks in VAL_DS], axis=0)[..., 0]
    AE_VAL_MAPS = compute_error_map(ACTIVE_MODEL, AE_VAL_IMAGES)
    assert AE_VAL_MAPS.shape == (len(AE_VAL_IMAGES), *CFG.image_size), (
        "error maps must be (N, H, W) with one plane per validation frame"
    )
    assert np.isfinite(AE_VAL_MAPS).all(), "error maps contain NaN or infinity"
    assert float(AE_VAL_MAPS.min()) >= 0.0, "reconstruction error cannot be negative"

In [ ]:
# OWNER: Adrian
def binarize_scores(scores: np.ndarray, threshold: float) -> np.ndarray:
    """Apply one scalar threshold to score maps.

    Inputs: float32 scores `(N, H, W)` and a threshold fixed on validation.
    Outputs: boolean masks `(N, H, W)`; True means anomaly.
    Contract: the threshold is finite and inside the observed validation score
    range; the output has the input shape; no post-processing is applied here.
    Pitfall: applying a threshold per frame is a different detector. One scalar
    for the whole split, fixed before test.
    """
    # Hint: np.asarray(scores) > threshold. Keep the comparison direction
    # explicit: high error or high anomaly score means anomaly.
    raise NotImplementedError("TODO[Adrian]: implement score thresholding")

In [ ]:
# OWNER: Adrian
def select_threshold(scores: np.ndarray, masks: np.ndarray, criterion: str) -> float:
    """Choose one anomaly threshold on validation only.

    Inputs: flattened validation scores and binary masks, plus the criterion name
    from `CFG.threshold_criterion` (for example a target F1 or a fixed FPR).
    Outputs: one finite float threshold.
    Contract: the threshold lies within the observed validation score range; the
    criterion and the resulting validation score are recorded; no test value is
    read inside this function.
    Pitfall: maximizing a metric on test is not threshold selection; it is test
    fitting. The test split is not an argument here on purpose.
    """
    # Hint: sweep candidate thresholds, compute the criterion with the same
    # metric definitions as 4.1, and return the best scalar. Record ties and the
    # selected validation value; do not import test data.
    raise NotImplementedError("TODO[Adrian]: implement validation-only threshold selection")


if CFG.approach == "autoencoder":
    AE_THRESHOLD = select_threshold(
        AE_VAL_MAPS.ravel(), AE_VAL_MASKS.ravel(), CFG.threshold_criterion
    )
    assert np.isfinite(AE_THRESHOLD), "the selected threshold must be finite"
    assert float(AE_VAL_MAPS.min()) <= AE_THRESHOLD <= float(AE_VAL_MAPS.max()), (
        "the threshold must lie inside the validation score range"
    )
    SELECTED_THRESHOLD = AE_THRESHOLD
    print(f"Threshold fixed on validation: {SELECTED_THRESHOLD}")
    AE_VAL_PRED = binarize_scores(AE_VAL_MAPS, AE_THRESHOLD)
    assert AE_VAL_PRED.shape == AE_VAL_MAPS.shape, "thresholding must preserve shape"
    assert AE_VAL_PRED.dtype == np.bool_, "thresholding must return a boolean mask"

In [ ]:
# OWNER: Adrian
def postprocess_mask(mask: np.ndarray, min_area: int) -> np.ndarray:
    """Remove small connected components from one predicted mask.

    Inputs: a boolean `(H, W)` mask and the minimum component area in pixels.
    Outputs: a boolean `(H, W)` mask with the same shape.
    Contract: components smaller than `min_area` are removed; the mask stays
    boolean; the operation is documented and applied identically to every frame.
    Pitfall: tuning `min_area` on test improves the table and invalidates it.
    Fix it on validation and report it with the threshold.
    """
    # Hint: scipy.ndimage.label and np.bincount to measure component areas; keep
    # or remove whole components. Do not use a morphological open with an
    # unrecorded kernel size.
    raise NotImplementedError("TODO[Adrian]: implement mask post-processing")


if CFG.approach == "autoencoder":
    assert CFG.postprocess_min_area is not None, (
        "Set CFG.postprocess_min_area on validation before post-processing"
    )
    AE_VAL_CLEAN = np.stack(
        [postprocess_mask(mask, CFG.postprocess_min_area) for mask in AE_VAL_PRED]
    )
    assert AE_VAL_CLEAN.shape == AE_VAL_PRED.shape, "post-processing must preserve shape"
    assert AE_VAL_CLEAN.dtype == np.bool_, "post-processing must return a boolean mask"

In [ ]:
# OWNER: Adrian
def analyze_error_regions(pred_mask: np.ndarray, true_mask: np.ndarray) -> dict[str, float]:
    """Count true positives, false positives, false negatives and their sizes.

    Inputs: predicted and ground-truth boolean masks `(H, W)`.
    Outputs: `tp`, `fp`, `fn`, `tn` pixel counts and the median connected-component
    area of each error type.
    Contract: counts are non-negative integers; `tp + fp + fn` counts every pixel
    that is anomalous in the prediction or the ground truth; no frame is dropped.
    Pitfall: a large FP blob and a single stray pixel are different failures;
    report component sizes, not only pixel totals.
    """
    # Hint: boolean intersection/difference for counts; scipy.ndimage.label for
    # component sizes. Keep FP and FN separate so the panel can show both.
    raise NotImplementedError("TODO[Adrian]: implement the FP/FN region study")


if CFG.approach == "autoencoder":
    AE_ERRORS = analyze_error_regions(AE_VAL_CLEAN[0], AE_VAL_MASKS[0].astype(bool))
    assert {"tp", "fp", "fn", "tn"} <= set(AE_ERRORS), (
        "error analysis must report tp, fp, fn and tn"
    )
    assert all(AE_ERRORS[name] >= 0 for name in ("tp", "fp", "fn", "tn")), (
        "pixel counts cannot be negative"
    )

In [ ]:
# OWNER: Adrian
def compute_auroc(scores: np.ndarray, masks: np.ndarray) -> float:
    """Compute pixel-level AUROC for anomaly scores.

    Inputs: flattened float scores and binary ground-truth masks.
    Outputs: AUROC in [0, 1].
    Contract: both classes are present; otherwise raise instead of returning a
    misleading 0.5; the anomaly ratio is reported beside the value.
    Pitfall: AUROC can look favourable when only a handful of pixels are
    anomalous. It is threshold-free, not imbalance-free.
    """
    # Hint: sklearn.metrics.roc_auc_score on flattened arrays; check that both
    # classes exist before calling it. Report the anomaly pixel ratio with it.
    raise NotImplementedError("TODO[Adrian]: implement pixel AUROC")


if CFG.approach == "autoencoder":
    AE_AUROC = compute_auroc(AE_VAL_MAPS.ravel(), AE_VAL_MASKS.ravel())
    assert 0.0 <= AE_AUROC <= 1.0, "AUROC must be in [0, 1]"
    print(f"Validation AUROC: {AE_AUROC} (report with anomaly coverage)")

In [ ]:
# OWNER: Adrian
def compute_aupro(scores: np.ndarray, masks: np.ndarray, fpr_cap: float) -> dict[str, float]:
    """Compute AUPRO and record the false-positive-rate integration cap.

    Inputs: flattened scores, binary masks and the FPR cap from
    `CFG.aupro_fpr_cap`.
    Outputs: `{"value": float, "fpr_cap": float}`.
    Contract: `0 < fpr_cap < 1`; the returned cap equals the argument; the value
    is in [0, 1]; a value without its cap is incomplete.
    Pitfall: comparing AUPRO values computed with different caps is meaningless.
    Keep the cap in every table cell and in the experiment record.
    """
    # Hint: per-region overlap curve, integrate only up to the declared FPR.
    # Return the cap in the result object instead of dropping it after plotting.
    raise NotImplementedError("TODO[Adrian]: implement AUPRO with its FPR cap")


if CFG.approach == "autoencoder":
    assert CFG.aupro_fpr_cap is not None, "Set CFG.aupro_fpr_cap before computing AUPRO"
    AE_AUPRO = compute_aupro(AE_VAL_MAPS.ravel(), AE_VAL_MASKS.ravel(), CFG.aupro_fpr_cap)
    assert AE_AUPRO["fpr_cap"] == CFG.aupro_fpr_cap, (
        "the AUPRO result must carry the cap it was computed with"
    )
    assert 0.0 <= AE_AUPRO["value"] <= 1.0, "AUPRO must be in [0, 1]"

### 4.3 DeepLabV3+ diagnostics and Seg-Grad-CAM — Ines · Emna

**Goal.** Add Boundary IoU, conditional mIoU, Seg-Grad-CAM and Integrated
Gradients for DeepLabV3+.

**Why it matters.** mIoU is meaningless on a binary target; Boundary IoU changes
with band width. A CAM without its target layer and class set cannot be checked.

**Feynman check.** The inspector marks the whole region and, separately, the
outline. The magnifying glass shows where the tool looked, not why the answer is
correct; pointing at pixels is a second, separate check.

**Watch out.** Report the band width with Boundary IoU. mIoU only after a verified
multi-class label map. The stride-4 low-level path bypasses ASPP and the
global-pooling branch is spatially constant; state both limits beside the CAM.
Record the IG baseline and step count.

**Your task.**
- [ ] [Ines · Emna] Verify the multi-class labels before enabling mIoU.
- [ ] [Ines · Emna] Fix the threshold on validation, then compute Boundary IoU with `CFG.boundary_iou_band`.
- [ ] [Ines · Emna] Produce Seg-Grad-CAM per ASPP branch and decoder stage; record each target layer.
- [ ] [Ines · Emna] Produce Integrated Gradients with the recorded baseline and step count.
- [ ] [Ines · Emna] State the ASPP and decoder limits in the interpretation.

In [ ]:
# OWNER: Ines · Emna
def load_multiclass_validation(cfg: Config) -> tuple[np.ndarray, np.ndarray]:
    """Load verified multi-class normal-road validation images and class ids.

    Inputs: the configuration and the pinned normal-road validation split.
    Outputs: float32 images `(N, H, W, 3)` in [0, 1] and int32 class ids
    `(N, H, W)`.
    Contract: no augmentation; the same spatial preprocessing as 2.2; class ids
    are the verified label map, with ignored ids excluded or marked.
    Pitfall: reusing the binary anomaly masks here would make mIoU a two-class
    score and hide every failed road class.
    """
    # Hint: reuse the validation records and decoder from 2.2, but read the
    # multi-class mask, not the binary target. Assert the class set against the
    # verified label map before returning.
    raise NotImplementedError("TODO[Todos]: load verified multi-class validation data")


if CFG.approach == "deeplabv3plus" and CFG.multiclass_labels_verified:
    assert CFG.segmentation_num_classes is not None, (
        "Set CFG.segmentation_num_classes after verifying the multi-class label map"
    )
    MULTI_IMAGES, MULTI_CLASSES = load_multiclass_validation(CFG)
    assert MULTI_IMAGES.shape[:3] == MULTI_CLASSES.shape, (
        "multi-class images and labels must share (N, H, W)"
    )
    assert int(MULTI_CLASSES.max()) < CFG.segmentation_num_classes, (
        "a class id is outside the declared class set"
    )

In [ ]:
# OWNER: Ines · Emna
def compute_miou(
    pred_classes: np.ndarray,
    true_classes: np.ndarray,
    num_classes: int,
    ignore_labels: tuple[int, ...],
) -> dict[str, Any]:
    """Compute per-class IoU and their mean for verified multi-class labels.

    Inputs: predicted and true class ids `(N, H, W)`, the class count and ignored
    ids. Outputs: `per_class` (class id -> IoU), `mean` and `classes`.
    Contract: `mean` is the mean of the per-class values, not a pixel average;
    ignored pixels are excluded; a failed class is not hidden by the mean.
    Pitfall: mIoU on a binary anomaly mask is not this metric. Enable it only
    when the multi-class labels are verified.
    """
    # Hint: confusion matrix over class ids, then IoU per class; ignore ids before
    # counting. np.nanmean over the per-class values, and keep the class set.
    raise NotImplementedError("TODO[Todos]: implement multi-class mIoU")


if CFG.approach == "deeplabv3plus" and CFG.multiclass_labels_verified:
    assert ACTIVE_MODEL is not None, "Train DeepLabV3+ before computing mIoU"
    MULTI_LOGITS = ACTIVE_MODEL(MULTI_IMAGES, training=False).numpy()
    MULTI_PRED = np.argmax(MULTI_LOGITS, axis=-1)
    MIOU = compute_miou(
        MULTI_PRED, MULTI_CLASSES, CFG.segmentation_num_classes, CFG.ignore_labels or ()
    )
    assert {"per_class", "mean", "classes"} <= set(MIOU), (
        "mIoU must report per-class values, the mean and the class set"
    )
    assert abs(MIOU["mean"] - float(np.nanmean(list(MIOU["per_class"].values())))) < 1e-9, (
        "mIoU must be the mean of per-class IoU, not a pixel average"
    )

In [ ]:
# OWNER: Ines · Emna
if CFG.approach == "deeplabv3plus":
    assert CFG.threshold_criterion is not None, (
        "Set CFG.threshold_criterion before fixing the DeepLabV3+ threshold"
    )
    DEEPLAB_VAL_IMAGES = np.concatenate([images.numpy() for images, _ in VAL_DS], axis=0)
    DEEPLAB_VAL_SCORES, DEEPLAB_VAL_MASKS = score_dataset(ACTIVE_MODEL, VAL_DS, "deeplabv3plus")
    DEEPLAB_THRESHOLD = select_threshold(
        DEEPLAB_VAL_SCORES.ravel(), DEEPLAB_VAL_MASKS.ravel(), CFG.threshold_criterion
    )
    SELECTED_THRESHOLD = DEEPLAB_THRESHOLD
    DEEPLAB_VAL_PRED = binarize_scores(DEEPLAB_VAL_SCORES, DEEPLAB_THRESHOLD)
    assert DEEPLAB_VAL_PRED.shape == DEEPLAB_VAL_MASKS.shape, (
        "thresholded predictions must align with validation masks"
    )
    assert np.isfinite(DEEPLAB_THRESHOLD), "the selected threshold must be finite"

In [ ]:
# OWNER: Ines · Emna
def compute_boundary_iou(
    pred_mask: np.ndarray, true_mask: np.ndarray, band_width: float
) -> dict[str, float]:
    """Compute Boundary IoU inside a band around the true and predicted edges.

    Inputs: boolean `(H, W)` masks and the band width from
    `CFG.boundary_iou_band`. Outputs: `{"value": float, "band_width": float}`.
    Contract: the value is in [0, 1]; the returned band width equals the argument;
    the width unit (pixels or fraction of the image diagonal) is written down.
    Pitfall: a Boundary IoU without its band width cannot be compared with any
    other value, including the same model at another width.
    """
    # Hint: scipy.ndimage.binary_dilation to build the boundary band, then
    # intersect it with the masks. Keep the width as both an input and an output.
    raise NotImplementedError("TODO[Todos]: implement Boundary IoU")


if CFG.approach == "deeplabv3plus":
    assert CFG.boundary_iou_band is not None, (
        "Set CFG.boundary_iou_band before computing Boundary IoU"
    )
    DEEPLAB_BOUNDARY = compute_boundary_iou(
        DEEPLAB_VAL_PRED[0],
        DEEPLAB_VAL_MASKS[0].astype(bool),
        CFG.boundary_iou_band,
    )
    assert DEEPLAB_BOUNDARY["band_width"] == CFG.boundary_iou_band, (
        "the result must carry the band width it was computed with"
    )
    assert 0.0 <= DEEPLAB_BOUNDARY["value"] <= 1.0, "Boundary IoU must be in [0, 1]"

In [ ]:
# OWNER: Ines · Emna
def seg_gradcam(
    model: keras.Model,
    image: np.ndarray,
    target_layer: str,
    target_pixels: tuple[tuple[int, int], ...],
) -> np.ndarray:
    """Compute Seg-Grad-CAM for a defined target pixel set.

    Inputs: a trained segmentation model, one float32 `(H, W, 3)` image, the
    decoder layer name and the target pixel coordinates.
    Outputs: a float32 heatmap `(H, W)`, normalized by a declared rule.
    Contract: finite heatmap at the input spatial size; the layer, target pixels
    and normalization are recorded; the input preprocessing matches inference.
    Pitfall: a heatmap over the wrong layer or class is a plausible picture with
    no meaning. Do not call it causal evidence.
    """
    # Hint: tf.GradientTape around the target layer output, gradient of the
    # selected score summed over target pixels, global-average the gradients,
    # weight the feature maps, ReLU and resize to the input. Record the layer.
    raise NotImplementedError("TODO[Todos]: implement Seg-Grad-CAM")


if CFG.approach == "deeplabv3plus":
    assert CFG.xai_target_layer and CFG.xai_target_pixel is not None, (
        "Set CFG.xai_target_layer and CFG.xai_target_pixel before Seg-Grad-CAM"
    )
    DEEPLAB_GRADCAM = seg_gradcam(
        ACTIVE_MODEL,
        DEEPLAB_VAL_IMAGES[0],
        CFG.xai_target_layer,
        (CFG.xai_target_pixel,),
    )
    assert DEEPLAB_GRADCAM.shape == DEEPLAB_VAL_IMAGES[0].shape[:2], (
        "the heatmap must match the input spatial size"
    )
    assert np.isfinite(DEEPLAB_GRADCAM).all(), "the heatmap contains NaN or infinity"

In [ ]:
# OWNER: Ines · Emna
def integrated_gradients(
    model: keras.Model,
    image: np.ndarray,
    target_pixel: tuple[int, int],
    baseline: np.ndarray,
    steps: int,
) -> np.ndarray:
    """Attribute one target score to the input pixels with Integrated Gradients.

    Inputs: a trained model, one float32 `(H, W, 3)` image, a target pixel, a
    baseline image of the same shape and a positive step count.
    Outputs: float32 attribution `(H, W, 3)` with the input shape.
    Contract: finite values; the baseline and step count are recorded; the target
    scalar is the score at `target_pixel`; preprocessing matches inference.
    Pitfall: changing the baseline or the step count changes the attribution.
    Results are not comparable across baselines.
    """
    # Hint: interpolate between the baseline and the image, sum gradients over
    # the target scalar, multiply by (image - baseline); tf.GradientTape in a
    # loop. Record the baseline name and steps beside the saved attribution.
    raise NotImplementedError("TODO[Todos]: implement Integrated Gradients")

In [ ]:
# OWNER: Ines · Emna
if CFG.approach == "deeplabv3plus":
    assert CFG.xai_baseline is not None and CFG.xai_baseline_name, (
        "Set CFG.xai_baseline and CFG.xai_baseline_name; IG is baseline-sensitive"
    )
    assert CFG.xai_ig_steps is not None and CFG.xai_ig_steps > 0, (
        "Set a positive CFG.xai_ig_steps and record it with the attribution"
    )
    assert CFG.xai_baseline.shape == DEEPLAB_VAL_IMAGES[0].shape, (
        "the baseline must have the same (H, W, 3) shape as the input"
    )
    DEEPLAB_IG = integrated_gradients(
        ACTIVE_MODEL,
        DEEPLAB_VAL_IMAGES[0],
        CFG.xai_target_pixel,
        CFG.xai_baseline,
        CFG.xai_ig_steps,
    )
    assert DEEPLAB_IG.shape == DEEPLAB_VAL_IMAGES[0].shape, (
        "the attribution must match the input shape"
    )
    assert np.isfinite(DEEPLAB_IG).all(), "the attribution contains NaN or infinity"

### 4.4 U-Net diagnostics, explanations and comparison — Hejer · Walid

**Goal.** Add HD95, Seg-Grad-CAM, Integrated Gradients and the CNN/U-Net
comparison.

**Why it matters.** Dice can hide a boundary that is far away; HD95 measures that
distance in pixels. A heatmap without a target and baseline cannot be reproduced.

**Feynman check.** One inspector compares the drawn outline with the true outline
and reports the distance in pixels; another points at the pixels that pushed the
answer. The pointing is evidence about the tool, not a better answer.

**Watch out.** HD95 is lower-is-better and not bounded in [0, 1]. State the empty
prediction/empty ground-truth rule. Seg-Grad-CAM uses decoder features; the
bottleneck is an ablation, not the primary result.

**Your task.**
- [ ] [Walid] Select the U-Net threshold on validation only.
- [ ] [Walid] Implement HD95 in pixels with the empty-mask rule.
- [ ] [Hejer · Walid] Produce Seg-Grad-CAM on a decoder layer and Integrated Gradients with a recorded baseline and step count.
- [ ] [Hejer · Walid] Fill the CNN/U-Net comparison from experiment records only.

In [ ]:
# OWNER: Walid
if CFG.approach == "unet":
    assert CFG.threshold_criterion is not None, (
        "Set CFG.threshold_criterion before fixing the U-Net threshold"
    )
    UNET_VAL_IMAGES = np.concatenate([images.numpy() for images, _ in VAL_DS], axis=0)
    UNET_VAL_SCORES, UNET_VAL_MASKS = score_dataset(ACTIVE_MODEL, VAL_DS, "unet")
    UNET_THRESHOLD = select_threshold(
        UNET_VAL_SCORES.ravel(), UNET_VAL_MASKS.ravel(), CFG.threshold_criterion
    )
    SELECTED_THRESHOLD = UNET_THRESHOLD
    UNET_VAL_PRED = binarize_scores(UNET_VAL_SCORES, UNET_THRESHOLD)
    assert UNET_VAL_PRED.shape == UNET_VAL_MASKS.shape, (
        "thresholded predictions must align with validation masks"
    )
    assert np.isfinite(UNET_THRESHOLD), "the selected threshold must be finite"

In [ ]:
# OWNER: Walid
def compute_hd95(
    pred_mask: np.ndarray,
    true_mask: np.ndarray,
    spacing: tuple[float, float],
    empty_policy: str,
) -> dict[str, Any]:
    """Compute the 95th percentile of the symmetric boundary distance.

    Inputs: boolean `(H, W)` masks, pixel spacing and the written empty-mask
    policy. Outputs: `value`, `units="pixels"`, `spacing` and `empty_policy`.
    Contract: the value is finite and non-negative; units are pixels; lower is
    better; the value is not bounded in [0, 1]; the empty policy is echoed.
    Pitfall: silently returning 0 for an empty prediction makes a missed object
    look perfect. Apply the rule from 1.4 and say which case occurred.
    """
    # Hint: scipy.ndimage.distance_transform_edt on the boundary masks, then
    # np.percentile(..., 95) over both directions. Multiply distances by spacing.
    raise NotImplementedError("TODO[Walid]: implement HD95 in pixels")


if CFG.approach == "unet":
    assert CFG.empty_mask_policy is not None, "Set CFG.empty_mask_policy before computing HD95"
    UNET_HD95 = compute_hd95(
        UNET_VAL_PRED[0],
        UNET_VAL_MASKS[0].astype(bool),
        (1.0, 1.0),
        CFG.empty_mask_policy,
    )
    assert UNET_HD95["units"] == "pixels", "HD95 must be reported in pixels"
    assert UNET_HD95["value"] >= 0.0, "HD95 cannot be negative"
    assert UNET_HD95["empty_policy"] == CFG.empty_mask_policy, (
        "HD95 must echo the empty-mask policy it used"
    )

In [ ]:
# OWNER: Hejer · Walid
if CFG.approach == "unet":
    assert CFG.xai_target_layer and CFG.xai_target_pixel is not None, (
        "Set CFG.xai_target_layer and CFG.xai_target_pixel before Seg-Grad-CAM"
    )
    UNET_GRADCAM = seg_gradcam(
        ACTIVE_MODEL,
        UNET_VAL_IMAGES[0],
        CFG.xai_target_layer,
        (CFG.xai_target_pixel,),
    )
    assert UNET_GRADCAM.shape == UNET_VAL_IMAGES[0].shape[:2], (
        "the heatmap must match the input spatial size"
    )
    assert np.isfinite(UNET_GRADCAM).all(), "the heatmap contains NaN or infinity"

In [ ]:
# OWNER: Hejer · Walid
if CFG.approach == "unet":
    assert CFG.xai_baseline is not None and CFG.xai_baseline_name, (
        "Set CFG.xai_baseline and CFG.xai_baseline_name; IG is baseline-sensitive"
    )
    assert CFG.xai_ig_steps is not None and CFG.xai_ig_steps > 0, (
        "Set a positive CFG.xai_ig_steps and record it with the attribution"
    )
    assert CFG.xai_baseline.shape == UNET_VAL_IMAGES[0].shape, (
        "the baseline must have the same (H, W, 3) shape as the input"
    )
    UNET_IG = integrated_gradients(
        ACTIVE_MODEL,
        UNET_VAL_IMAGES[0],
        CFG.xai_target_pixel,
        CFG.xai_baseline,
        CFG.xai_ig_steps,
    )
    assert UNET_IG.shape == UNET_VAL_IMAGES[0].shape, "the attribution must match the input shape"
    assert np.isfinite(UNET_IG).all(), "the attribution contains NaN or infinity"

#### 4.4.1 CNN baseline versus U-Net

Fill this only from experiment records. The CNN baseline is the same builder with
`skip_connections=False`; it is not a separate approach row in the final table.

| Model | Precision | Recall | F1 | IoU | Dice | Comment |
|---|---|---|---|---|---|---|
| CNN encoder-decoder baseline | — | — | — | — | — | — |
| U-Net | — | — | — | — | — | — |

> Write your observation here: what did the skip connections change, and what
> changed for a reason other than the skip path? Tie every number to a record.

### 4.5 Error-map panel and interpretation — Khaled · Adrian

**Goal.** Show the image, true mask, predicted mask and reconstruction-error map
beside a written interpretation, and attribute the mean reconstruction error with
Integrated Gradients.

**Why it matters.** A visual without the failure case beside it becomes a
showcase. The panel is for inspection, not for promotion.

**Feynman check.** The inspector lays the photograph, the correct circle, the
drawn circle and the surprise map on one desk, then writes what went wrong.

**Watch out.** The error map is a model output, not an explanation. Integrated
Gradients explains the scalar mean reconstruction error; neither is Seg-Grad-CAM.
Do not use either to choose a model.

**Your task.**
- [ ] [Khaled · Adrian] Implement the four-panel figure with a comment line.
- [ ] [Khaled · Adrian] Choose a frame that includes a failure, not only a success.
- [ ] [Khaled · Adrian] Write the interpretation in the caption, not in the title.
- [ ] [Khaled · Adrian] Implement reconstruction-error Integrated Gradients with the recorded baseline and steps.
- [ ] [Khaled · Adrian] State that the error map is not a causal explanation.

In [ ]:
# OWNER: Khaled · Adrian
def plot_error_map_panel(
    image: np.ndarray,
    true_mask: np.ndarray,
    pred_mask: np.ndarray,
    error_map: np.ndarray,
    threshold: float,
    comment: str,
) -> Any:
    """Build the four-panel autoencoder error figure.

    Inputs: one image `(H, W, 3)`, true and predicted masks `(H, W)`, the error
    map `(H, W)`, the threshold and a written comment.
    Outputs: a Matplotlib `Figure` with four axes, in the order image, true mask,
    predicted mask, error map, plus one caption line with the comment.
    Contract: exactly four axes; masks use nearest display; the threshold is
    visible; the caption is the team's observation, not a generated claim.
    Pitfall: a pretty panel without the failure frame is cherry-picking. Keep the
    frame id and the threshold in the caption.
    """
    # Hint: plt.subplots(1, 4), imshow per panel, interpolation="nearest" for
    # masks, fig.text or fig.suptitle for the comment. Return fig, do not show it.
    raise NotImplementedError("TODO[Todos]: implement the error-map panel")


if CFG.approach == "autoencoder":
    assert CFG.postprocess_min_area is not None, "Post-process the mask before the panel"
    ERROR_PANEL = plot_error_map_panel(
        AE_VAL_IMAGES[0],
        AE_VAL_MASKS[0].astype(bool),
        AE_VAL_CLEAN[0],
        AE_VAL_MAPS[0],
        AE_THRESHOLD,
        "",
    )
    assert len(ERROR_PANEL.axes) == 4, "the error panel must have exactly four axes"

In [ ]:
# OWNER: Khaled · Adrian
def reconstruction_integrated_gradients(
    model: keras.Model,
    image: np.ndarray,
    baseline: np.ndarray,
    steps: int,
) -> np.ndarray:
    """Attribute the mean reconstruction error to input pixels with IG.

    Inputs: a trained autoencoder, one float32 `(H, W, 3)` image, a baseline of
    the same shape and a positive step count.
    Outputs: float32 attribution `(H, W, 3)` with the input shape.
    Contract: the target scalar is the mean reconstruction error over all pixels;
    the baseline name and step count are recorded; values are finite; the
    preprocessing matches inference.
    Pitfall: this attribution is not the per-pixel error map and must not be
    labelled as one. A different baseline changes the result.
    """
    # Hint: reconstruct inside a GradientTape, reduce the error to one mean
    # scalar, interpolate baseline -> image, average the gradients and scale by
    # (image - baseline). Record the baseline name and steps with the artifact.
    raise NotImplementedError("TODO[Todos]: implement reconstruction-error Integrated Gradients")


if CFG.approach == "autoencoder":
    assert CFG.xai_baseline is not None and CFG.xai_baseline_name, (
        "Set CFG.xai_baseline and CFG.xai_baseline_name; IG is baseline-sensitive"
    )
    assert CFG.xai_ig_steps is not None and CFG.xai_ig_steps > 0, (
        "Set a positive CFG.xai_ig_steps and record it with the attribution"
    )
    assert CFG.xai_baseline.shape == AE_VAL_IMAGES[0].shape, (
        "the baseline must have the same (H, W, 3) shape as the input"
    )
    AE_IG = reconstruction_integrated_gradients(
        ACTIVE_MODEL, AE_VAL_IMAGES[0], CFG.xai_baseline, CFG.xai_ig_steps
    )
    assert AE_IG.shape == AE_VAL_IMAGES[0].shape, "the attribution must match the input shape"
    assert np.isfinite(AE_IG).all(), "the attribution contains NaN or infinity"

### 4.6 Shared XAI panel, error analysis and comparison — Todos

**Goal.** Score the frozen test split once, build the shared explanation panel,
write the limitation and fill the comparison only from records.

**Why it matters.** The test split is opened once; the comparison table is a
summary of evidence, not a place to estimate. The selection rationale comes from
normal-road validation.

**Feynman check.** The sealed exam is opened, marked once and filed. The contact
sheet explains the mistakes; the shift board compares tools without changing the
marks.

**Watch out.** Do not tune the threshold, post-processing or target layer on the
test split. XAI never promotes a model. A missing record means an empty cell, not
an estimate.

**Your task.**
- [ ] [Todos] Score the test split once and keep the per-frame distribution.
- [ ] [Todos] Build the shared XAI panel and the TP/FP/FN panel.
- [ ] [Todos] Run the deletion/insertion and pointing-game faithfulness checks with their baselines.
- [ ] [Todos] Fill the comparison table only from experiment records.
- [ ] [Todos] Write the selection rationale from normal-road validation.
- [ ] [Todos] State the XAI limitation and keep all three candidates in the registry.

In [ ]:
# OWNER: Todos
# The single test-touch point. Threshold and post-processing are already fixed.
assert CFG.approach is not None and ACTIVE_MODEL is not None, (
    "Train the active approach before scoring the test split"
)
assert SELECTED_THRESHOLD is not None, "Fix the threshold on validation before test"
assert SPLIT["frozen"] is True, "the test split must still be frozen"
assert CFG.empty_mask_policy is not None, (
    "Write the empty-mask rule in 4.1.1 before scoring the test split"
)
assert CFG.ignore_labels is not None, (
    "Set CFG.ignore_labels from the Section 1 audit before scoring"
)

TEST_SCORES, TEST_MASKS = score_dataset(ACTIVE_MODEL, TEST_DS, CFG.approach)
TEST_PRED = binarize_scores(TEST_SCORES, SELECTED_THRESHOLD)
assert TEST_SCORES.shape == TEST_MASKS.shape, "test scores and masks must align"
assert TEST_PRED.shape == TEST_MASKS.shape, "test predictions and masks must align"

PER_IMAGE_METRICS = [
    compute_binary_metrics(pred, true.astype(bool))
    for pred, true in zip(TEST_PRED, TEST_MASKS, strict=True)
]
TEST_METRICS = aggregate_metrics(PER_IMAGE_METRICS)
assert {"precision", "recall", "f1", "iou", "dice"} <= set(TEST_METRICS), (
    "the aggregate must expose all five required metrics"
)
assert all(0.0 <= value <= 1.0 for value in TEST_METRICS.values()), (
    "every binary metric must be in [0, 1]"
)
assert abs(TEST_METRICS["f1"] - TEST_METRICS["dice"]) < 1e-9, (
    "F1 must equal Dice for the same binary mask"
)
assert TEST_METRICS["iou"] <= TEST_METRICS["dice"] + 1e-9, (
    "IoU cannot exceed Dice for the same mask"
)

In [ ]:
# OWNER: Todos
def plot_xai_panel(
    image: np.ndarray,
    true_mask: np.ndarray,
    pred_mask: np.ndarray,
    explanation: np.ndarray,
    comment: str,
) -> Any:
    """Build the shared four-panel XAI figure.

    Inputs: one image, true and predicted masks, a heatmap or error map and a
    written comment. Outputs: a `Figure` with four axes in the order image, true
    mask, predicted mask, explanation, plus one caption line.
    Contract: exactly four axes; the explanation is labelled as Seg-Grad-CAM,
    Integrated Gradients or error map; the caption is a written observation; no
    title claims causation or a model ranking.
    Pitfall: one panel type for all methods hides that an error map is not an
    attribution. The label must match the method that produced the array.
    """
    # Hint: plt.subplots(1, 4), imshow with nearest for masks, a colour map for
    # the explanation, fig.text for the comment. Return fig and record the method
    # name with the artifact.
    raise NotImplementedError("TODO[Todos]: implement the shared XAI panel")

In [ ]:
# OWNER: Todos
if CFG.approach == "autoencoder":
    XAI_PANEL = plot_xai_panel(
        AE_VAL_IMAGES[0],
        AE_VAL_MASKS[0].astype(bool),
        AE_VAL_CLEAN[0],
        AE_VAL_MAPS[0],
        "",
    )
elif CFG.approach == "deeplabv3plus":
    XAI_PANEL = plot_xai_panel(
        DEEPLAB_VAL_IMAGES[0],
        DEEPLAB_VAL_MASKS[0].astype(bool),
        DEEPLAB_VAL_PRED[0],
        DEEPLAB_GRADCAM,
        "",
    )
else:
    XAI_PANEL = plot_xai_panel(
        UNET_VAL_IMAGES[0],
        UNET_VAL_MASKS[0].astype(bool),
        UNET_VAL_PRED[0],
        UNET_GRADCAM,
        "",
    )
assert len(XAI_PANEL.axes) == 4, "the shared XAI panel must have exactly four axes"

In [ ]:
# OWNER: Todos
def xai_faithfulness_check(
    model: keras.Model,
    image: np.ndarray,
    mask: np.ndarray,
    explanation: np.ndarray,
    approach: str,
    baseline: np.ndarray,
    steps: int,
    region_size: int,
) -> dict[str, Any]:
    """Run deletion/insertion and the pointing game on one explained frame.

    Inputs: the active model, one image, its binary mask, the explanation map,
    the approach id, the IG baseline, a positive step count and a positive
    region size in pixels.
    Outputs: `deletion_curve`, `insertion_curve`, `deletion_auc`,
    `insertion_auc`, the random-order and center-prior AUCs, `pointing_hit_rate`,
    `pointing_random_hit_rate`, `steps` and `region_size`.
    Contract: each curve has `steps + 1` points; AUC values are finite; the
    pointing fields are `None` for the autoencoder, which has no class score.
    Pitfall: ranking regions on the test frame and then tuning the explanation
    with that result turns a report-only check into test fitting.
    """
    # Hint: rank pixels by |explanation|; delete top regions to the baseline and
    # insert them into the baseline, measuring the target after each step. Repeat
    # with random and center priors; for segmentation, check the argmax in the mask.
    raise NotImplementedError("TODO[Todos]: implement XAI faithfulness checks")

In [ ]:
# OWNER: Todos
if CFG.approach == "autoencoder":
    FAITH_IMAGE = AE_VAL_IMAGES[0]
    FAITH_MASK = AE_VAL_MASKS[0]
    FAITH_EXPLANATION = AE_VAL_MAPS[0]
elif CFG.approach == "deeplabv3plus":
    FAITH_IMAGE = DEEPLAB_VAL_IMAGES[0]
    FAITH_MASK = DEEPLAB_VAL_MASKS[0]
    FAITH_EXPLANATION = DEEPLAB_GRADCAM
else:
    FAITH_IMAGE = UNET_VAL_IMAGES[0]
    FAITH_MASK = UNET_VAL_MASKS[0]
    FAITH_EXPLANATION = UNET_GRADCAM

In [ ]:
# OWNER: Todos
assert CFG.approach is not None and ACTIVE_MODEL is not None, (
    "Train the active approach before the faithfulness checks"
)
assert CFG.xai_baseline is not None and CFG.xai_baseline_name, (
    "Set CFG.xai_baseline and CFG.xai_baseline_name before the faithfulness checks"
)
assert CFG.xai_faithfulness_steps is not None and CFG.xai_faithfulness_steps > 0, (
    "Set a positive CFG.xai_faithfulness_steps"
)
assert CFG.xai_region_size is not None and CFG.xai_region_size > 0, (
    "Set a positive CFG.xai_region_size in pixels"
)
FAITHFULNESS = xai_faithfulness_check(
    ACTIVE_MODEL,
    FAITH_IMAGE,
    FAITH_MASK.astype(bool),
    FAITH_EXPLANATION,
    CFG.approach,
    CFG.xai_baseline,
    CFG.xai_faithfulness_steps,
    CFG.xai_region_size,
)

In [ ]:
# OWNER: Todos
assert {"deletion_curve", "insertion_curve", "pointing_hit_rate"} <= set(FAITHFULNESS), (
    "the faithfulness result must expose both curves and the pointing field"
)
assert len(FAITHFULNESS["deletion_curve"]) == CFG.xai_faithfulness_steps + 1, (
    "the deletion curve must have one point per step plus the start"
)
assert len(FAITHFULNESS["insertion_curve"]) == CFG.xai_faithfulness_steps + 1, (
    "the insertion curve must have one point per step plus the start"
)
assert all(
    np.isfinite(FAITHFULNESS[key])
    for key in (
        "deletion_auc",
        "insertion_auc",
        "random_deletion_auc",
        "random_insertion_auc",
        "center_deletion_auc",
        "center_insertion_auc",
    )
), "every faithfulness AUC must be finite"
if CFG.approach == "autoencoder":
    assert FAITHFULNESS["pointing_hit_rate"] is None, (
        "the pointing game is not applicable to the autoencoder"
    )
else:
    assert 0.0 <= FAITHFULNESS["pointing_hit_rate"] <= 1.0, (
        "the pointing hit rate must be a fraction"
    )

In [ ]:
# OWNER: Todos
# Use the same region study as 4.2, now on the frozen test frame.
TEST_ERRORS = analyze_error_regions(TEST_PRED[0], TEST_MASKS[0].astype(bool))
assert {"tp", "fp", "fn", "tn"} <= set(TEST_ERRORS), "the error panel must report tp, fp, fn and tn"
assert all(TEST_ERRORS[name] >= 0 for name in ("tp", "fp", "fn", "tn")), (
    "error counts cannot be negative"
)

#### 4.6.1 Error-analysis panel

Fill one row per inspected frame from the test split. Keep the worst frame, not
only the best one. Obstacle size is a measured pixel area; write the unit.

| Frame | TP | FP | FN | Obstacle size (px) | Comment |
|---|---|---|---|---|---|
| | | | | | |
| | | | | | |
| | | | | | |

#### 4.6.2 Comparison table

Fill only from experiment records. Latency is a measured value with the hardware
recorded; it is not an accuracy metric. Parameters are counted from the built
model. F1 equals Dice for a binary mask.

| Approach | Precision | Recall | F1 | IoU | Dice | Latency (ms) | Parameters |
|---|---|---|---|---|---|---|---|
| `deeplabv3plus` | — | — | — | — | — | — | — |
| `autoencoder` | — | — | — | — | — | — | — |
| `unet` | — | — | — | — | — | — | — |

#### 4.6.3 Selection rationale

> Write the rationale here. Select from normal-road validation only; keep all
> three candidate SavedModels and manifests in the registry; `selected.json` is
> an alias, not a replacement. State the native metric, uncertainty, threshold,
> failure modes and the reason for the choice. Do not put a RoadAnomaly number in
> the selection column.

- [ ] [Todos] The rationale uses normal-road validation and the scorecard, not the benchmark.
- [ ] [Todos] All three candidates remain in the registry; the alias is documented.
- [ ] [Todos] XAI is report-only and does not appear in `selected.json`.

#### 4.6.4 Required limitation

Heatmaps and error maps are **not causal explanations** and do not replace
segmentation metrics. Seg-Grad-CAM and Integrated Gradients show correlation and
are baseline-sensitive; the reconstruction-error map shows where the model is
surprised, not why. No explanation certifies safety or correctness, and none
promotes a model.

> Write the limitation for the frame you inspected, in your own words, beside the
> artifact. Do not reuse the paragraph above as the only interpretation.

## 5. Inference and Deployment Helpers (optional) — Khaled · Adrian

**Goal.** Provide three optional helpers: single-image inference with an overlay,
a measured latency/FPS report and a checkpoint-reload consistency check.

**Why it matters.** A demo can preprocess differently from evaluation and serve a
different answer. Latency quoted without hardware and warm-up is not a
measurement.

**Feynman check.** The inspector hands one marked photograph to the dispatcher and
times how long the desk takes per photograph. The number is written with the desk
and the machine, not as a slogan.

**Watch out.** Warm up before timing, repeat the measurement, report the device
and batch size, and keep every claim tied to the measured numbers. Reload the
exported checkpoint and compare outputs within `CFG.reload_tolerance`.

**Your task.**
- [ ] [Khaled · Adrian] Implement single-image inference and the overlay.
- [ ] [Khaled · Adrian] Measure latency and FPS with warm-up, repeats and hardware.
- [ ] [Khaled · Adrian] Reload the checkpoint and assert the output difference.
- [ ] [Todos] Keep Streamlit, FastAPI, Docker and video work in the separate code.

### 5.1 Single-image inference — Khaled · Adrian

Run one image through the active model with the validation-fixed threshold and
return the score map, mask and overlay.

In [ ]:
# OWNER: Khaled · Adrian
def predict_single_image(model: keras.Model, image_path: Path, threshold: float) -> dict[str, Any]:
    """Run one image through the active model and build an overlay.

    Inputs: the trained model, one readable image path and the validation-fixed
    threshold. Outputs: `score_map`, `mask`, `overlay`, `threshold` and
    `input_shape`; score map and mask are `(H, W)`, overlay is `(H, W, 3)`.
    Contract: preprocessing matches 2.2 and 3.3; the returned threshold is the
    one passed in; no result is written to the experiment record from here.
    Pitfall: a different resize or colour order silently changes the demo answer.
    Reuse the training preprocessing, do not copy it.
    """
    # Hint: PIL/tf.io to load, the same resize and normalization as 2.2, expand
    # the batch axis, score, threshold, then alpha-blend the mask on the image.
    # Return arrays; leave file writing to the deployment code.
    raise NotImplementedError("TODO[Todos]: implement single-image inference")


assert CFG.approach is not None and ACTIVE_MODEL is not None, (
    "Train the active approach before single-image inference"
)
assert SELECTED_THRESHOLD is not None, "Fix the threshold on validation first"
assert CFG.image_size is not None, "Set CFG.image_size before inference"
DEMO_RECORD = NORMAL_MANIFEST["records"][0]
DEMO_PATH = repo_path(Path(NORMAL_MANIFEST["root"]) / Path(DEMO_RECORD["image"]))
SINGLE = predict_single_image(ACTIVE_MODEL, DEMO_PATH, SELECTED_THRESHOLD)
assert {"score_map", "mask", "overlay", "threshold", "input_shape"} <= set(SINGLE), (
    "single-image inference must return the documented keys"
)
assert SINGLE["score_map"].shape == CFG.image_size, "score map must be (H, W)"
assert SINGLE["mask"].shape == CFG.image_size, "mask must be (H, W)"
assert SINGLE["overlay"].shape == (*CFG.image_size, 3), "overlay must be (H, W, 3)"
assert SINGLE["threshold"] == SELECTED_THRESHOLD, "echo the threshold that produced the mask"

### 5.2 Latency and FPS measurement — Khaled · Adrian

Warm up before timing, repeat the measurement and record the hardware. Latency
is not an accuracy metric.

In [ ]:
# OWNER: Khaled · Adrian
def measure_latency(
    model: keras.Model,
    sample_batch: tf.Tensor,
    *,
    warmup: int = 5,
    repeats: int = 20,
) -> dict[str, Any]:
    """Measure per-batch latency and throughput on the current device.

    Inputs: the model, one batched sample, a warm-up count and a repeat count.
    Outputs: `latency_ms_mean`, `latency_ms_median`, `latency_ms_p95`, `fps`,
    `batch_size`, `warmup`, `repeats` and `hardware`.
    Contract: warm-up runs are excluded; at least one timed repeat; all timing
    values are positive; `hardware` is a non-empty description of the device.
    Pitfall: timing the first call includes graph tracing. Timing without the
    device name produces a number nobody can reproduce.
    """
    # Hint: run `warmup` untimed calls, then time `repeats` calls with
    # time.perf_counter, synchronize if needed, and report median/p95 and batch
    # size. Read the device from tf.config.list_physical_devices.
    raise NotImplementedError("TODO[Todos]: implement the latency/FPS helper")


assert ACTIVE_MODEL is not None, "Train the active approach before measuring latency"
LATENCY = measure_latency(ACTIVE_MODEL, next(iter(VAL_DS))[0])
assert LATENCY["latency_ms_median"] > 0.0, "median latency must be positive"
assert LATENCY["fps"] > 0.0, "measured FPS must be positive"
assert LATENCY["repeats"] > 0, "at least one timed repeat is required"
assert LATENCY["hardware"], "latency without hardware is not a measurement"

### 5.3 Checkpoint-reload consistency — Khaled · Adrian

Reload the best checkpoint and compare its output with the in-memory model
within the configured tolerance.

In [ ]:
# OWNER: Khaled · Adrian
def check_checkpoint_reload(
    model: keras.Model,
    checkpoint_path: Path,
    sample_batch: tf.Tensor,
    tolerance: float,
) -> dict[str, Any]:
    """Reload a checkpoint and compare its output with the in-memory model.

    Inputs: the trained model, a checkpoint path, one batched sample and the
    tolerance from `CFG.reload_tolerance`.
    Outputs: `max_abs_diff`, `tolerance` and `passed`.
    Contract: the reloaded model uses the same preprocessing; `passed` is True
    exactly when `max_abs_diff <= tolerance`; the tolerance is echoed.
    Pitfall: comparing probabilities with logits hides a head change. Compare the
    same output tensor the serving path will use.
    """
    # Hint: keras.models.load_model(checkpoint_path), run both models on the same
    # preprocessed batch, and take np.max(np.abs(a - b)). Do not re-fit.
    raise NotImplementedError("TODO[Todos]: implement the checkpoint-reload check")


assert CFG.checkpoint_path is not None, (
    "Set CFG.checkpoint_path to the best checkpoint before the reload check"
)
RELOAD = check_checkpoint_reload(
    ACTIVE_MODEL,
    repo_path(CFG.checkpoint_path),
    next(iter(VAL_DS))[0],
    CFG.reload_tolerance,
)
assert RELOAD["tolerance"] == CFG.reload_tolerance, (
    "the reload result must echo the tolerance it used"
)
assert RELOAD["max_abs_diff"] <= CFG.reload_tolerance, (
    "reloaded model output differs beyond CFG.reload_tolerance"
)
assert RELOAD["passed"] is True, "the reload consistency check did not pass"

### 5.4 Separate delivery code

Streamlit, FastAPI, Docker and video are covered by the team's separate code. This
notebook creates no API, no container, no UI and no video artifact, and it does
not render a placeholder explanation. The only contract this notebook owes the
delivery layer is the score map, the threshold that produced the mask, the model
identity and the measured latency with its hardware.

> Write your observation here: which of the four delivery surfaces consumes this
> notebook's artifacts, and which artifact digest does it pin?

## 6. Final Checklist — Todos

**Goal.** Tick only what a file, command or experiment record proves.

**Why it matters.** A ticked box is a claim. The requirements matrix defines the
evidence bar, and a README sentence is not evidence.

**Feynman check.** The shift supervisor signs the handover only after seeing the
case file, the sealed exam result and the logbook entry, not after hearing that
the inspector is confident.

**Watch out.** Do not tick a box because the code exists. Tick it when the run is
reproducible from the commit, config, manifest, split hash, command and hardware.

**Your task.**
- [ ] [Todos] Tick each line only after the evidence exists, and link it in the report.
- [ ] [Todos] Leave an unticked line unticked; it is the honest state.
- [ ] [Todos] Re-read the Scope Lock before adding a model, metric or method.

- [ ] [Todos] Dataset audit complete; binary target and empty-mask rule written in 1.4.
- [ ] [Todos] Normal-road manifest, licence and split hash pinned; the run YAML in `configs/` references both; zero scene overlap.
- [ ] [Todos] Loaders: image float32 in [0, 1], mask int32 in {0, 1}, nearest mask resize.
- [ ] [Todos] Augmentation is train-only, synchronized, and applied after the split.
- [ ] [Ines] DeepLabV3+ builder, ASPP/dilated-convolution justification and target layer recorded.
- [ ] [Emna] Encoder weight source, licence, preprocessing and frozen/fine-tuned regime recorded.
- [ ] [Emna] DeepLabV3+ training and experiment tracking reproducible.
- [ ] [Khaled] Autoencoder trained on normal-road images only; reconstruction range recorded.
- [ ] [Adrian] Error maps, validation threshold, post-processing, FP/FN study, AUROC and AUPRO with its cap.
- [ ] [Hejer] CNN encoder-decoder baseline trained and reported through the shared metrics.
- [ ] [Walid] U-Net trained with early stopping and the best checkpoint reloaded; Dice/IoU/HD95 reported.
- [ ] [Todos] Loss and monitoring metrics defined once; no second metric implementation.
- [ ] [Ines · Emna] Boundary IoU band recorded; mIoU only if multi-class labels are verified.
- [ ] [Ines · Emna] DeepLabV3+ Seg-Grad-CAM per ASPP/decoder stage and Integrated Gradients with recorded baseline/steps.
- [ ] [Hejer · Walid] U-Net Seg-Grad-CAM on a decoder layer and Integrated Gradients with recorded baseline/steps.
- [ ] [Khaled · Adrian] Error-map panel and written interpretation, including a failure frame.
- [ ] [Todos] Deletion/insertion and pointing-game results with random and center baselines.
- [ ] [Todos] Test split scored once; per-frame distribution and empty-mask handling reported.
- [ ] [Todos] One experiment record per run with commit, config, normal-road manifest, split hash, command and hardware.
- [ ] [Todos] Comparison table filled only from records; latency and parameters evidenced.
- [ ] [Todos] Selection rationale uses normal-road validation; all three candidates stay in the registry.
- [ ] [Todos] XAI limitation stated; explanations are report-only and never promote a model.
- [ ] [Khaled · Adrian] Latency/FPS measured with warm-up, repeats and hardware; reload check within tolerance.
- [ ] [Todos] Streamlit, FastAPI, Docker and video work lives in the separate code, not this notebook.
- [ ] [Todos] Outputs cleared, `nbstripout --install` used, `make lint type test` green.
- [ ] [Todos] `data/credits.txt` attribution travels with every published frame.

If a line cannot be ticked, leave it blank and record the reason in the experiment
record or the requirements matrix.